# Procurement Title Clustering Pipeline
**Dataset-agnostic | spaCy NLP | TF-IDF | Agglomerative + HDBSCAN | Auto Noise Reassignment | Category Mapping**

## Step 1 · Imports

In [1]:
import re, warnings, json, math
from pathlib import Path
from collections import Counter, defaultdict
from itertools import combinations

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.sparse import issparse
import hdbscan
import spacy
from spacy.lang.en.stop_words import STOP_WORDS as SPACY_STOP

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

warnings.filterwarnings('ignore')
print("All imports OK")

All imports OK


## Step 2 · Configuration
Update the values below for a new dataset. Everything downstream runs automatically.

In [2]:
INPUT_FILE   = 'IL-P2P-PO.xlsx'
SHEET_NAME   = None
TITLE_COL    = 'PO_TITLE'
OUTPUT_FILE  = 'Clustered_Output_final_testing_2.xlsx'

EXTRA_COLS   = ['BASICVALUE', 'isTendered',
                'PO_NUMBER', 'VENDOR',
                'VENDOR_NAME']

# ── Clustering method: 'auto', 'agglomerative', or 'hdbscan'
CLUSTER_METHOD = 'auto'   # 'auto' picks based on dataset size

DISTANCE_THRESHOLD   = 0.85
MAX_BUCKET           = 500
HDBSCAN_MIN_CLUSTER  = 3
HDBSCAN_MIN_SAMPLES  = 2

# ── Noise reassignment thresholds
NOISE_THRESHOLD      = 8
REASSIGN_SIM_CUTOFF  = 0.40
SINGLETON_MERGE_CUTOFF = 0.40

# ── Product-keyword split guards (uncomment pairs to prevent merges)
PRODUCT_SPLIT_PAIRS = [
    # ('fire extinguisher', 'fire alarm'),
    # ('ups battery',       'ups system'),
    # ('transformer oil',   'transformer repair'),
]

# ── Central noise store (accumulates across dataset runs)
CENTRAL_NOISE_FILE = 'central_noise_store.json'

print("Configuration loaded.")

Configuration loaded.


## Step 3 · NLP Setup & Stop-Word List

In [3]:
nlp = spacy.blank('en')
nlp.add_pipe('lemmatizer', config={'mode': 'lookup'})
nlp.initialize()

UNIVERSAL_STOPS = {
    # Grammar
    'of','at','to','for','and','in','the','by','from','with','a','an','on','is',
    'are','be','was','were','as','into','its','or','per','no','new',
    # Procurement action words (not product names)
    'supply','supplies','supplied','provision','providing','purchase','procurement',
    'order','orders','contract','contracts','tender','bid','bidding','proposal',
    'annual','yearly','rate','rates','basis','during','period','year','lpg','manpower','assistance','visit','kg','kgs','inch',
    'regarding','related','various','including','etc','misc',"sply","supp","inst","instln","system","machine","Suply","Suply","Instal",
    'finish','quality','joint','bulk','automation','assistance','manpower','management','station','hpcl','gmo','ez','pro',
    'emergency','hills','design','kit','warehouse','licenses','additive','change','refuelling',"accessories",'data','instrument',
    'online','sampling','web','warehouse','kva','ird','loactions','kva','meter','sale','area','sales','outlet',
    'job','jobs','work','works','working','sup','ranchi','centre','petroleum','non','jagdalpur','tokheim',
    # Execution verbs
    'installation','install','testing','commissioning','erection','fabrication',
    'maintenance','repair','repairs','replacement','revamping','modification',
    'upgradation','renovation','service','services','servicing','construction',
    'loading','unloading','fixing','execution','carrying','shifting','region','sa','plains','varo','kota','ne','states','tokheim,'
    'du',
    # Generic filler
    'item','items','spares','spare','equipment','equipments','material','materials',
    'requirement','requirements','complete','comprehensive','allied','associated',
    'required','miscellaneous','existing','new','various','filling','fill',
    # Org/time noise
    'ltd','pvt','limited','india','corporation','rfp','eoi',
    'po','pr','amc','camc','fy','fy24','fy25','fy26',
    'project','facility','plant','unit','office','head','district',
    'regional','headquarters','retail','vendor','contractor','consultant','boom','set','laying','line','testing','seal','panel','load','unload','loading','unloading','handle','handling','overhaul','overhauling','arm','shed','level','card','check','vacuum','sheet','charge','support','consumable','pt','gd','cyl','cyls','kirloskar','repair','protection','weight','ally','allied','modification','sitc','activity','pressure','barrier','switch','port','operation','motor','software','fab','commissioning','training','csr','jatni','meter','platform','fixing','building','augmentation','instl','compliance','pump','enabling','kosan','crossover','etp','sludge','power','cabinet','analysis','providing','transfer','oil','fw','aluminium','suppy','suppl','honeywell','cavern','hydraulic','block','accessories','detector','weld','welding','overhead','type','breaker','comm','controller','distribution','connection','fitting','engineer','urgent','communication','required','mula','operator','box','mt','vision','preparation','transformer','m3/hr','led','wagon','conkote','elt','installation','plan','old','wc','nrv','campaign','module','modules','strip','officer','officers','hospitality','removal',
    'q','integrated','canopy','work','plain','usn','allied','hills','torna','hoga','evcs','cluster2','shro','hill','thane','sa','sbp','pre','fab','light','nellore','vjrro','dist','mansa','fazilka','bhawanipatna','kashmir','shimla','nh','mprdc','rajnandgaon','dw','chittoor','bidar','shirdi','ahmednagar','baihata','tata','karkardooma','deoghar','jalgaon','jalna','bhilwara','hailakandi','agomoni','bachupally','lokhra','brahmaputra','nagaon','unakoti','panbari','lengpu','dergaon','mandia','dawki','jorhat','bhawanipur','mokokchung','marbaniang','arasikere','zunheboto','tamranga','naginimora','miao','matabari','damangiri','umkiang','mawmyngkreng','budge','badarpur','laluk','rangia','thilixu','sakardara','anand','hoshangabad','amravati','dobaspet','kondalapur','vishramgrih','vishramghar','araff','precast','island','jalpaiguri','alipurduar','bandra','chandkheda','hatia','2583','2587','2588','2598','2470','2473','2304','2341','2342','2125','1671','699','700','444','491','510','3','167','168','169','2179','2180','tml','water','handling','handle','point','points','type','testing','panel','ii','locations','location','additional','varat','rajasthan','small','group','nov','mar','load','unloading','ga','cluster','wb','insta','installation','firozabad','sa','vro','shimla','anand','baroda','pro','budan','cbg','instal','frzbd','agra','tirur','kanur','sultanpur','alwar','dist','relate','plains','plain','fatepur','approval','asf','urban','rural','company','esa','zone','schedule','vjrro','daman','dnh',
    'water','zone','asf','house','chandigarh','supply','instln','comm','location','ams','kotyark','depot','integration','installation','commissioning','activity','sites','kurukshetra','deoria','hiring','relate','nellore','noose','set','tml','afff','mnr','gj','products','tirupati','instt'
}

ALL_STOPS = UNIVERSAL_STOPS | {w.lower() for w in SPACY_STOP}
AT_CAPTURED_STOPS: set = set()

def load_central_noise() -> set:
    if Path(CENTRAL_NOISE_FILE).exists():
        with open(CENTRAL_NOISE_FILE) as f:
            data = json.load(f)
        noise = set(data.get('noise_words', []))
        print(f"  Loaded {len(noise)} words from central noise store.")
        return noise
    return set()

def save_central_noise():
    existing = load_central_noise()
    combined = existing | AT_CAPTURED_STOPS | domain_noise
    with open(CENTRAL_NOISE_FILE, 'w') as f:
        json.dump({'noise_words': sorted(combined), 'total_count': len(combined)}, f, indent=2)
    print(f"  Central noise store updated: {len(combined)} words saved.")

central_noise = load_central_noise()
ALL_STOPS.update(central_noise)
print(f"  Total effective stop words after central load: {len(ALL_STOPS)}")

def capture_and_remove_at_words(text: str) -> str:
    def capture_chain(text):
        pattern = re.compile(
            r'(?:@\s*|(?<!\w)at\s+)'
            r'('
                r'\S+'
                r'(?:\s+(?:and|&)\s+\S+)*'
            r')',
            re.IGNORECASE
        )
        for m in pattern.finditer(text):
            chain = m.group(1)
            parts = re.split(r'\s+(?:and|&)\s+', chain, flags=re.IGNORECASE)
            for part in parts:
                for word in part.split():
                    word = re.sub(r'[^a-z]', '', word.lower())
                    if word and len(word) > 1:
                        AT_CAPTURED_STOPS.add(word)
                        ALL_STOPS.add(word)
        return text
    text = capture_chain(text)
    text = re.sub(r'(?:@\s*|(?<!\w)at\s+)\S+(?:\s+(?:and|&)\s+\S+)*', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\b(?:to|from)\s+\S+\b', ' ', text, flags=re.IGNORECASE)
    return text.strip()

def spacy_preprocess(text: str) -> str:
    text = capture_and_remove_at_words(str(text))
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text.lower())
    doc  = nlp(text)
    tokens = []
    for t in doc:
        lemma = t.lemma_.lower()
        if lemma in ALL_STOPS or len(lemma) <= 1:
            continue
        if lemma.isdigit():
            continue
        stripped = re.sub(r'\d+$', '', lemma)
        if len(stripped) <= 1:
            continue
        tokens.append(stripped)
    return ' '.join(tokens)

print(f"spaCy pipeline ready. Total effective stop words: {len(ALL_STOPS)}")

  Loaded 1424 words from central noise store.
  Total effective stop words after central load: 2128
spaCy pipeline ready. Total effective stop words: 2128


## Step 4 · Noun Phrase / Product Keyword Extraction

In [4]:
ACTION_WORDS = {
    'supply','installation','maintenance','repair','testing','commissioning',
    'procurement','purchase','service','services','replacement','erection',
    'revamping','renovation','modification','fabrication','construction',
    'shifting','fixing','carrying','execution','providing','operation',
    'inspection','calibration','cleaning','painting','coating','welding',
    'checking','certification','renewal','audit','assessment','survey',
    'disposal','decommissioning','dismantling','upgrading','upgradation',
    'loading','unloading','handling','storage','transportation','delivery',
}

def extract_product_keywords(raw_title: str) -> list:
    text   = re.sub(r'[^a-zA-Z0-9\s]', ' ', str(raw_title).lower())
    doc    = nlp(text)
    tokens = [
        t.lemma_.lower()
        for t in doc
        if t.lemma_.lower() not in ALL_STOPS
        and t.lemma_.lower() not in ACTION_WORDS
        and len(t.lemma_) > 1
        and not t.lemma_.isdigit()
        and not re.search(r'\d', t.lemma_)
    ]
    phrases = list(tokens)
    for i in range(len(tokens) - 1):
        phrases.append(f"{tokens[i]} {tokens[i+1]}")
    for i in range(len(tokens) - 2):
        phrases.append(f"{tokens[i]} {tokens[i+1]} {tokens[i+2]}")
    return phrases

def has_conflicting_product_keywords(kw_set_a: set, kw_set_b: set) -> bool:
    for kw_a, kw_b in PRODUCT_SPLIT_PAIRS:
        tokens_a = set(kw_a.split())
        tokens_b = set(kw_b.split())
        if tokens_a.issubset(kw_set_a) and tokens_b.issubset(kw_set_b):
            return True
        if tokens_b.issubset(kw_set_a) and tokens_a.issubset(kw_set_b):
            return True
    return False

def detect_domain_noise(titles: pd.Series, freq_threshold: float = 0.55) -> set:
    total  = len(titles)
    counts = Counter()
    for title in titles:
        tokens = set(spacy_preprocess(str(title)).split())
        counts.update(tokens)
    noise  = {tok for tok, cnt in counts.items() if cnt / total >= freq_threshold}
    if noise:
        print(f"  Auto-detected domain noise tokens (freq >= {freq_threshold*100:.0f}%): {sorted(noise)}")
    return noise

print("Noun phrase extractor & merge-guard ready.")

MERGE_GROUPS = [
    ['Ethanol Esy', 'Esy','Esy2025'],
    ['Esy2024 C3','Esy2024 C5','Esy C1','Esy2025 C1','Esy2024 C1']
]

def flatten_merge_map(groups: list) -> dict:
    merge_map = {}
    for group in groups:
        if not group:
            continue
        canonical = group[0]
        for name in group[1:]:
            merge_map[name] = canonical
    return merge_map

def apply_manual_merges(df: pd.DataFrame) -> pd.DataFrame:
    merge_map = flatten_merge_map(MERGE_GROUPS)
    if not merge_map:
        print("  No manual merges defined — skipping.")
        return df
    df['Standard Name'] = df['Standard Name'].replace(merge_map)
    name_to_min_id = df.groupby('Standard Name')['Cluster ID'].min()
    df['Cluster ID'] = df['Standard Name'].map(name_to_min_id)
    print(f"  Manual merges applied: {len(merge_map)} name(s) remapped.")
    return df

print("Helper functions ready.")

Noun phrase extractor & merge-guard ready.
Helper functions ready.


## Step 5 · Load Data

In [5]:
print(f"Loading: {INPUT_FILE}")
df_raw = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME or 0)
print(f"  Raw rows: {len(df_raw):,}  |  Columns: {list(df_raw.columns)}")

assert TITLE_COL in df_raw.columns, f"Column '{TITLE_COL}' not found. Available: {list(df_raw.columns)}"

keep_cols  = [TITLE_COL] + [c for c in EXTRA_COLS if c in df_raw.columns]
df         = df_raw[keep_cols].copy()
df.columns = [TITLE_COL] + [c for c in EXTRA_COLS if c in df_raw.columns]
df         = df.dropna(subset=[TITLE_COL]).reset_index(drop=True)

for c in df.columns:
    if c != TITLE_COL:
        df[c] = df[c].fillna('—')

print(f"  After dropna: {len(df):,} rows")
print(f"\nSample titles:")
for t in df[TITLE_COL].head(8):
    print(f"  {str(t)[:90]}")

Loading: IL-P2P-PO.xlsx
  Raw rows: 56,106  |  Columns: ['BASICVALUE', 'PO Created By Name', 'PO Created By ID', 'CURRENCY', 'DOC_DATE', 'PO Doc Type', 'EXCH_RATE', 'FIRST_APPROVED_DATE', 'GOVT_POLICY', 'GOVT_POLICY_CODE', 'GST Type', 'isGlobalDomestic', 'isTendered', 'LANDEDVALUE', 'MAKE_IN_INDIA', 'MAKE_IN_INDIA_CODE', 'PO_NUMBER', 'PO_TITLE', 'Purchase Group', 'Purchase Preference', 'PURCH_PREF_CODE', 'Purchase Group Description', 'TAXVALUE', 'Tender Category', 'TENDER_NO', 'VENDOR', 'VENDOR_MSE_DESC', 'Vendor MSE Flag', 'VENDOR_NAME', 'VENDOR_PAN', 'Day Month Num', 'Month Abbr', 'Fiscal Year', 'First Approved Date', 'Month SID', 'PO Placement Mode', 'isMII', 'Material', 'MATERIAL_CODE', 'isNomination', 'PO Doc Type Description', 'Tender Type', 'TENDER_TYPE_CD', 'PO Placement Mode Code', 'NOM_REASON_CD', 'NOMINATION_REASON', 'ESTIMATE', 'PR_SBU_CODE', 'SBU', 'Ex Target Value', 'GEM Contract Number', 'CAPEX_OPEX_FLAG', 'PO Approver ID', 'PO Approver Name', 'Basic Value', 'Landed Valu

## Step 6 · Preprocessing & Auto Noise Detection

In [6]:
print("Step 1 · Auto-detecting domain noise ...")
domain_noise = detect_domain_noise(df[TITLE_COL])
ALL_STOPS.update(domain_noise)
print(f"  Total effective stop words: {len(ALL_STOPS)}")

print("\nStep 2 · Preprocessing titles with spaCy ...")
df['_clean']   = df[TITLE_COL].apply(spacy_preprocess)
print(f"  Done. Sample:")
for raw, clean in zip(df[TITLE_COL].head(5), df['_clean'].head(5)):
    print(f"    {str(raw)[:60]!r:62s} -> {clean!r}")

print("\nStep 3 · Extracting product keyword phrases ...")
df['_keywords'] = df[TITLE_COL].apply(extract_product_keywords)

global_kw_freq = Counter()
for kws in df['_keywords']:
    global_kw_freq.update(kws)

print(f"  Top 20 product phrases across dataset:")
for phrase, cnt in global_kw_freq.most_common(20):
    print(f"    {phrase:<35s} {cnt:>5}")

Step 1 · Auto-detecting domain noise ...
  Total effective stop words: 2128

Step 2 · Preprocessing titles with spaCy ...
  Done. Sample:
    'THIRD PARTY INSPECTION- BELLARY RO'                           -> 'party inspection'
    'OLOA 59788'                                                   -> 'oloa'
    'ETHANOL CONTRACT ESY 2023-24 C1'                              -> 'ethanol esy'
    'ETHANOL CONTRACT ESY 2023-24 CY 1'                            -> 'ethanol esy cy'
    'ETHANOL CONTRACT ESY 2023-24 CY 1'                            -> 'ethanol esy cy'

Step 3 · Extracting product keyword phrases ...
  Top 20 product phrases across dataset:
                                        10134
    civil                                3537
    sor                                  2204
    ethanol                              1812
    electrical                           1803
    civil sor                            1695
    esy                                  1389
    cylinder              

## Step 7 · TF-IDF Vectorisation

In [7]:
n = len(df)

def augment_text(clean_text: str, keywords: list) -> str:
    top_multi = [kw for kw in keywords if ' ' in kw][:3]
    return clean_text + ' ' + ' '.join(top_multi)

df['_augmented'] = [
    augment_text(c, k)
    for c, k in zip(df['_clean'], df['_keywords'])
]

min_df_val = max(2, int(n * 0.0008))
print(f"TF-IDF settings: ngram=(1,1), min_df={min_df_val}, max_df=0.90, sublinear_tf=True")

vec = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 1),
    min_df=3,
    max_df=0.90,
    sublinear_tf=True,
)
X = vec.fit_transform(df['_augmented'])

print(f"  Matrix shape: {X.shape}  (rows x features)")
print(f"  Sparsity: {1 - X.nnz / (X.shape[0]*X.shape[1]):.1%}")

norms   = np.asarray(X.sum(axis=1)).ravel()
mask_nz = norms > 0
mask_z  = ~mask_nz
print(f"  Non-empty rows: {mask_nz.sum():,}  |  Empty rows: {mask_z.sum():,}")

TF-IDF settings: ngram=(1,1), min_df=44, max_df=0.90, sublinear_tf=True
  Matrix shape: (55279, 4078)  (rows x features)
  Sparsity: 100.0%
  Non-empty rows: 49,722  |  Empty rows: 5,557


In [8]:
X_fulldf = X.copy()

## Step 8 · Clustering (KMeans bucketing + Agglomerative)

In [9]:
import numpy as np
import pandas as pd
from sklearn.cluster import MiniBatchKMeans, AgglomerativeClustering
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_distances
from collections import deque
import time

t0 = time.time()
print(f"DISTANCE_THRESHOLD = {DISTANCE_THRESHOLD}")
print(f"MAX_BUCKET         = {MAX_BUCKET}")

titles_series    = df['PO_TITLE'].astype(str)
first_occurrence = titles_series.drop_duplicates(keep='first')
unique_titles    = first_occurrence.values
unique_row_idx   = first_occurrence.index.to_numpy()
print(f"Unique titles: {len(unique_titles):,}  (of {len(df):,} total rows)")

X      = X_fulldf[unique_row_idx]
X_norm = normalize(X, norm='l2')
print(f"TF-IDF matrix: {X_norm.shape}  nnz={X_norm.nnz:,}  [{time.time()-t0:.1f}s]")

queue   = deque([np.arange(len(unique_titles))])
buckets = []

while queue:
    idx = queue.popleft()
    if len(idx) <= MAX_BUCKET:
        buckets.append(idx)
        continue
    n_splits = min(max(2, len(idx) // MAX_BUCKET + 1), len(idx) // 2)
    km = MiniBatchKMeans(n_clusters=n_splits, random_state=42,
                         batch_size=min(1024, len(idx)), n_init=3)
    sub_labels = km.fit_predict(X_norm[idx])
    children   = [idx[sub_labels == g] for g in range(n_splits) if (sub_labels == g).any()]
    if len(children) <= 1 or max(len(c) for c in children) == len(idx):
        chunks = np.array_split(idx, max(2, len(idx) // MAX_BUCKET))
        for chunk in chunks:
            if len(chunk) > 0:
                queue.append(chunk)
    else:
        for child in children:
            queue.append(child)

bucket_sizes = [len(b) for b in buckets]
print(f"Buckets: {len(buckets):,}  max:{max(bucket_sizes)}  mean:{np.mean(bucket_sizes):.1f}  [{time.time()-t0:.1f}s]")

final_labels   = np.full(len(unique_titles), -1, dtype=int)
cluster_offset = 0

for i, idx in enumerate(buckets):
    if len(idx) == 1:
        final_labels[idx] = cluster_offset
        cluster_offset   += 1
        continue
    sub         = X_norm[idx].toarray()
    dist_matrix = cosine_distances(sub)
    sub_labels  = AgglomerativeClustering(
        n_clusters=None,
        distance_threshold=DISTANCE_THRESHOLD,
        metric='precomputed',
        linkage='complete',
    ).fit_predict(dist_matrix)
    final_labels[idx]  = sub_labels + cluster_offset
    cluster_offset    += sub_labels.max() + 1
    if (i + 1) % 1000 == 0:
        print(f"  {i+1:,}/{len(buckets):,} buckets  {cluster_offset:,} clusters  [{time.time()-t0:.1f}s]")

print(f"Clustering done [{time.time()-t0:.1f}s]  Total clusters: {cluster_offset:,}")

title_to_cluster = dict(zip(unique_titles, final_labels))
df['Cluster ID'] = (
    df['PO_TITLE'].astype(str).map(title_to_cluster).fillna(-1).astype(int)
)

print(f"\nTotal time: {time.time()-t0:.1f}s")
vc = df['Cluster ID'].value_counts()
print(f"\nCluster size distribution:")
print(f"  Singletons (1):  {(vc==1).sum():>6,}")
print(f"  Small    (2-5):  {((vc>=2)&(vc<=5)).sum():>6,}")
print(f"  Medium  (6-20):  {((vc>=6)&(vc<=20)).sum():>6,}")
print(f"  Large    (>20):  {(vc>20).sum():>6,}")
method = 'agglomerative'

DISTANCE_THRESHOLD = 0.85
MAX_BUCKET         = 500
Unique titles: 36,802  (of 55,279 total rows)
TF-IDF matrix: (36802, 4078)  nnz=68,428  [0.1s]
Buckets: 1,625  max:500  mean:22.6  [14.3s]
  1,000/1,625 buckets  6,978 clusters  [17.8s]
Clustering done [18.2s]  Total clusters: 7,949

Total time: 18.3s

Cluster size distribution:
  Singletons (1):   4,115
  Small    (2-5):   2,584
  Medium  (6-20):     841
  Large    (>20):     409


## Step 9 · Automatic Noise Reassignment

In [10]:
from sklearn.preprocessing import normalize as sk_normalize
from collections import Counter, defaultdict
import numpy as np
import time

t9 = time.time()
print("Running automatic noise reassignment ...")

X_norm_full    = sk_normalize(X_fulldf, norm='l2', copy=True)
current_labels = df['Cluster ID'].values.copy()
lbl_to_rows    = defaultdict(list)
for row_i, lbl in enumerate(current_labels):
    lbl_to_rows[lbl].append(row_i)
lbl_to_rows = {lbl: np.array(rows) for lbl, rows in lbl_to_rows.items()}

lbl_counts   = {lbl: len(rows) for lbl, rows in lbl_to_rows.items()}
large_labels = [lbl for lbl, cnt in lbl_counts.items() if cnt >= NOISE_THRESHOLD]
small_labels = [lbl for lbl, cnt in lbl_counts.items() if 0 < cnt < NOISE_THRESHOLD]
print(f"  Large clusters (>={NOISE_THRESHOLD} rows): {len(large_labels):,}")
print(f"  Small clusters (<{NOISE_THRESHOLD} rows): {len(small_labels):,}")

def sparse_centroid(row_indices):
    sub  = X_norm_full[row_indices]
    mean = np.asarray(sub.mean(axis=0)).ravel()
    norm = np.linalg.norm(mean)
    return mean / norm if norm > 0 else mean

centroid_lbls   = np.array(large_labels)
centroid_matrix = np.vstack([sparse_centroid(lbl_to_rows[lbl]) for lbl in large_labels])
print(f"  Centroid matrix: {centroid_matrix.shape}  [{time.time()-t9:.1f}s]")

keywords_by_row = df['_keywords'].tolist()
lbl_to_kw_set = {}
for lbl, rows in lbl_to_rows.items():
    kw_set = set()
    for ri in rows:
        kw_set.update(keywords_by_row[ri])
    lbl_to_kw_set[lbl] = kw_set

reassigned    = 0
kept_small    = 0
label_updates = {}

for lbl in small_labels:
    rows = lbl_to_rows[lbl]
    if len(rows) == 0:
        continue
    small_vec = sparse_centroid(rows)
    sims      = centroid_matrix.dot(small_vec)
    best_idx  = int(sims.argmax())
    best_sim  = float(sims[best_idx])
    best_lbl  = int(centroid_lbls[best_idx])
    if best_sim < REASSIGN_SIM_CUTOFF:
        kept_small += 1
        continue
    small_kw = lbl_to_kw_set.get(lbl, set())
    large_kw = lbl_to_kw_set.get(best_lbl, set())
    if has_conflicting_product_keywords(small_kw, large_kw):
        kept_small += 1
        continue
    label_updates[lbl] = best_lbl
    reassigned += 1

if label_updates:
    new_labels = current_labels.copy()
    for old_lbl, new_lbl in label_updates.items():
        new_labels[lbl_to_rows[old_lbl]] = new_lbl
    df['Cluster ID'] = new_labels

print(f"\n  Reassigned : {reassigned:,} small clusters absorbed into larger ones")
print(f"  Retained   : {kept_small:,} small clusters kept (too distinct or conflict)")
print(f"  Final clusters: {df['Cluster ID'].nunique():,}")

print("\nStep 6b · Merging singleton <-> singleton pairs ...")
current_labels = df['Cluster ID'].values.copy()
lbl_to_rows2   = defaultdict(list)
for row_i, lbl in enumerate(current_labels):
    lbl_to_rows2[lbl].append(row_i)
lbl_to_rows2    = {lbl: np.array(rows) for lbl, rows in lbl_to_rows2.items()}
lbl_counts2     = {lbl: len(rows) for lbl, rows in lbl_to_rows2.items()}
singleton_lbls  = [lbl for lbl, cnt in lbl_counts2.items() if cnt == 1]
merged_singletons = 0

if len(singleton_lbls) > 1:
    s_rows  = np.array([lbl_to_rows2[lbl][0] for lbl in singleton_lbls])
    S       = X_norm_full[s_rows]
    parent  = {lbl: lbl for lbl in singleton_lbls}
    already_merged = set()
    BATCH   = 500
    n_s     = len(singleton_lbls)
    for start in range(0, n_s, BATCH):
        end        = min(start + BATCH, n_s)
        batch_vecs = S[start:end]
        sims_block = np.asarray(batch_vecs.dot(S.T).todense())
        for bi, gi in enumerate(range(start, end)):
            lbl_a = singleton_lbls[gi]
            if lbl_a in already_merged:
                continue
            row_sims     = sims_block[bi].copy()
            row_sims[gi] = -1.0
            best_gi      = int(row_sims.argmax())
            best_sim     = float(row_sims[best_gi])
            if best_sim < SINGLETON_MERGE_CUTOFF:
                continue
            lbl_b = singleton_lbls[best_gi]
            if lbl_b in already_merged or lbl_b == lbl_a:
                continue
            kw_a = lbl_to_kw_set.get(lbl_a, set())
            kw_b = lbl_to_kw_set.get(lbl_b, set())
            if has_conflicting_product_keywords(kw_a, kw_b):
                continue
            already_merged.add(lbl_a)
            already_merged.add(lbl_b)
            parent[lbl_b] = lbl_a
            merged_singletons += 1

    if merged_singletons:
        new_labels2 = df['Cluster ID'].values.copy()
        for lbl_b, lbl_a in parent.items():
            if lbl_a != lbl_b:
                new_labels2[lbl_to_rows2[lbl_b]] = lbl_a
        df['Cluster ID'] = new_labels2

remaining_singletons = (df['Cluster ID'].value_counts() == 1).sum()
print(f"  Singleton pairs merged : {merged_singletons:,}")
print(f"  Remaining singletons   : {remaining_singletons:,}")
print(f"  Final clusters         : {df['Cluster ID'].nunique():,}")
print(f"  Total time (Cell 9)    : {time.time()-t9:.1f}s")

Running automatic noise reassignment ...
  Large clusters (>=8 rows): 931
  Small clusters (<8 rows): 7,018
  Centroid matrix: (931, 4078)  [0.3s]

  Reassigned : 881 small clusters absorbed into larger ones
  Retained   : 6,137 small clusters kept (too distinct or conflict)
  Final clusters: 7,068

Step 6b · Merging singleton <-> singleton pairs ...
  Singleton pairs merged : 84
  Remaining singletons   : 3,660
  Final clusters         : 6,984
  Total time (Cell 9)    : 19.8s


## Step 10 · Cluster Naming

In [11]:
print("Generating standard cluster names ...")

EXTRA_SKIP = {
    'with','this','that','from','have','will','been','they','were',
    'hpcl','plant','depot','under','services','service','supply','supplies',
    'work','works','misc','various','items','item','contract','contracts',
    'annual','maintenance','repair','repairs','installation','testing',
    'procurement','purchase','providing','provision','amc','camc',
    'sor','bro','nro','sro','gro','mro','rro','cro','kro','dro','pro','fro',
    'group','groups','small','large','zone','zones',
    'north','south','east','west','central',
    'new','old','existing','general','other',
}

def make_standard_name(cluster_id: int) -> str:
    cluster_rows = df[df['Cluster ID'] == cluster_id]
    n = len(cluster_rows)
    phrase_row_count = Counter()
    for kws in cluster_rows['_keywords']:
        for p in set(kws):
            if p.strip() and len(p.strip()) > 1:
                phrase_row_count[p.strip()] += 1
    MAJORITY_THRESHOLD = 0.50
    qualified = {p: c for p, c in phrase_row_count.items() if c / n >= MAJORITY_THRESHOLD}
    if qualified:
        best_multi  = sorted([(c, p) for p, c in qualified.items() if ' ' in p], reverse=True)
        best_single = sorted([(c, p) for p, c in qualified.items() if ' ' not in p], reverse=True)
        if best_multi:
            return best_multi[0][1].title()
        elif best_single:
            return best_single[0][1].title()
        else:
            return "MIXED — Needs Review"
    word_count = Counter()
    for _, row in cluster_rows.iterrows():
        raw   = str(row[TITLE_COL])
        if not raw or raw == 'nan':
            continue
        words = re.sub(r'[^a-zA-Z\s]', ' ', raw).lower().split()
        for w in words:
            if len(w) > 3 and w not in EXTRA_SKIP:
                word_count[w] += 1
    if word_count:
        top    = sorted(word_count.items(), key=lambda x: -x[1])
        chosen = []
        for w, _ in top:
            if not any(w in c or c in w for c in chosen):
                chosen.append(w)
            if len(chosen) == 2:
                break
        return ' '.join(w.title() for w in chosen)
    for _, row in cluster_rows.iterrows():
        raw = str(row[TITLE_COL]).strip()
        if raw and raw.lower() != 'nan':
            return raw[:50].title()
    return f'Cluster {cluster_id}'

_vc = df['Cluster ID'].value_counts().reset_index()
_vc.columns = ['old_id', 'sz']
_vc = _vc.sort_values('sz', ascending=False).reset_index(drop=True)
_vc['new_id'] = _vc.index + 1
id_map = dict(zip(_vc['old_id'], _vc['new_id']))
df['Cluster ID'] = df['Cluster ID'].map(id_map)

if 'PO_NUMBER' in df.columns:
    po_num = pd.to_numeric(df['PO_NUMBER'], errors='coerce')
    df = (df.assign(_po_sort=po_num)
            .sort_values(['Cluster ID', '_po_sort'], ascending=[True, False])
            .drop(columns=['_po_sort'])
            .reset_index(drop=True))
else:
    df = df.sort_values('Cluster ID').reset_index(drop=True)

df['Standard Name'] = df['Cluster ID'].apply(make_standard_name)

name_to_min_id = df.groupby('Standard Name')['Cluster ID'].min()
df['Cluster ID'] = df['Standard Name'].map(name_to_min_id)

_vc2 = df['Cluster ID'].value_counts().reset_index()
_vc2.columns = ['old_id', 'sz']
_vc2 = _vc2.sort_values('sz', ascending=False).reset_index(drop=True)
_vc2['new_id'] = _vc2.index + 1
id_map2 = dict(zip(_vc2['old_id'], _vc2['new_id']))
df['Cluster ID'] = df['Cluster ID'].map(id_map2)
df = df.sort_values('Cluster ID').reset_index(drop=True)
print(f"  After same-name merge: {df['Cluster ID'].nunique():,} clusters")

df = apply_manual_merges(df)

final_counts = df['Cluster ID'].value_counts()
print(f"  Standard names generated.")
print(f"  Final clusters : {df['Cluster ID'].nunique():,}")
print(f"  Singletons     : {(final_counts == 1).sum():,}")
print(f"  Largest cluster: {final_counts.max():,}")
print("\nSample cluster names:")
for _, r in df.drop_duplicates('Cluster ID').head(12)[['Cluster ID','Standard Name']].iterrows():
    print(f"  [{r['Cluster ID']:>4}] {r['Standard Name']}")

save_central_noise()

Generating standard cluster names ...
  After same-name merge: 5,594 clusters
  Manual merges applied: 6 name(s) remapped.
  Standard names generated.
  Final clusters : 5,594
  Singletons     : 2,273
  Largest cluster: 4,050

Sample cluster names:
  [   1] Ethanol Rajasthan
  [   2] Civil Sor
  [   3] Electrical
  [   4] Cylinder
  [   5] Civil
  [   6] Ethanol
  [   7] Mechanical
  [   8] Safety
  [   9] Dg
  [  10] Housekeeping
  [  11] Engine
  [  12] Pipeline
  Loaded 1424 words from central noise store.
  Central noise store updated: 1424 words saved.


In [12]:
# Save clustering checkpoint — run this once
CHECKPOINT_FILE = 'clustering_checkpoint.pkl'
df.to_pickle(CHECKPOINT_FILE)
print(f"Checkpoint saved: {len(df):,} rows, {df['Cluster ID'].nunique():,} clusters")

Checkpoint saved: 55,279 rows, 5,594 clusters


## Step 10b · Category / Sub Category Mapping
Matches each cluster against CATEGORY_MAP using **both Standard Name and all raw titles in the cluster** for much higher coverage. Falls back to 'Unmapped' only when no token overlap exists.

In [13]:
# category_mapping.py — edit CATEGORY_MAP and re-run this only
import pickle, re
import pandas as pd
from openpyxl import Workbook
# ... your imports ...

CHECKPOINT_FILE = 'clustering_checkpoint.pkl'
df = pd.read_pickle(CHECKPOINT_FILE)
print(f"Loaded checkpoint: {len(df):,} rows, {df['Cluster ID'].nunique():,} clusters")



Loaded checkpoint: 55,279 rows, 5,594 clusters


In [20]:
# =============================================================
# MASTER CATEGORY MAP
# Each sub-category phrase is matched against the cluster's
# Standard Name AND all raw PO titles in the cluster.
# Add new sub-category phrases here to reduce unmapped rate.
# =============================================================


CATEGORY_MAP = {
    "Corporate Services": [
        "Housekeeping",
        "Gardening",
        "Plumbing",
        "Electrical Maintenance",
        "Electrician",
        "AMC",
        "Canteen",
        "Consumables",
        "Stationery",
        "Vehicle Hiring",
        "Flower Arrangement",
        "Water Care Therapy",
        "GreenCo Consultancy",
        "Canteen Survey",
        "Gym Maintenance",
        "Aerobic Trainer",
        "Gym Equipment",
        "Massage Chair",
        "Television",
        "Furniture",
        "Air Conditioner",
        "HVAC",
        "Dishwasher",
        "Event Management",
        "Sports",
        "Catering",
        "Corporate Functions",
        "Gift Items",
        "Banners",
        "Flex Printing",
        "Samavesh Kits",
        "Decorative Items",
        "Printing",
        "Diwali Gifts",
        "Dry Fruits",
        "Training",
        "HR Events",
        "Recruitment",
        "Advertisements",
        "Insurance",
        "Software",
        "Software Licenses",
        "Hardware",
        "Hardware AMC",
        "Laptop",
        "Desktop",
        "Printer",
        "Digital Initiatives",
        "IS Strategy",
        "O&M Outsourcing",
        "Bottling Assistance",
        "LPG Web Portal",
        "Web Portal Development",
        "Contract Engineers",
        "ASF Manpower",
        "Vehicle Procurement",
        "Bitumen Handling Plant",
        "TFM RCD",
        "HP Pay",
        "OT Plus",
        "Loyalty Payments",
        "ITPS",
        "SMS Services",
        "Call Center",
        "SOP Audit",
        "CHP Audit",
        "Live Fire Training",
        "Biofuels",
        "Cow Dung",
        "Biomass",
        "Nitrogen Tower",
        "LED Installation",
        "Highway Assistance",
        "SAP HANA",
        "SAP Licenses",
        "Consultancy",
        "DGR Security",
        "Grass Cutting",    # "Grass Cut" SN is 2-token but stop word stripped
    "Grass Cut",        # keep raw
    "House Keeping",    # "House Keeping" SN match (2 tokens)
    "Housekeeping",
    "SOP Board",        # "Sop Board" SN match (2 tokens — stop word 'for' not in it)
    "SOP Items",
    "Diwali Sweets",    # "Diwali Sweet" SN match
    "Dryfruits Diwali",
    "TV Civil",         # "Tv" SN → corpus: "civil m&r tv1" → it's a civil area code
    "Shutter Painting", # "Shutter Paint" SN match
    "Wall Painting",
    "Lube Branding",
    "Architechtural",   # typo variant — add as-is so tokenizer matches
    "Architectural",
    "Architechtural Services",
    "Retail Outlet Works",  # "Retail Outlets" SN → corpus: "m&r at retail outlets"
    "CNG Retail Work",
    "Year Contract",    # "Yr" SN → corpus: "yrs work contract", "yrs service contract"
    "Multi Year Contract",
    ],

    "Mechanical": [
        "Steel",
        "SS Tanks",
        "Valves",
        "Pipes",
        "Pipe Fittings",
        "Flanges",
        "Steel Plates",
        "Pipe Rack",
        "Extension Works",
        "Loading Arms",
        "Unloading Arms",
        "Air Receivers",
        "Tankage",
        "Structural Works",
        "ASF Filters",
        "RVI",
        "Pylons",
        "High Mast",
        "Flexible Pipes",
        "Canopy",
        "MSV",
        "Safety Items",
        "Roof Drain System",
        "Aluminum Dome",
        "Aircraft Refueller",
        "Refueller Chassis",
        "Mechanical Works",
        "Pipeline Works"
    ],

    "Electrical & Instrumentation": [
        "Retail Automation",
        "Dispensing Units",
        "STP",
        "VRS",
        "EV Chargers",
        "LED",
        "Carousel",
        "Electrical Works",
        "Pumps",
        "LPG Compressors",
        "CCTV",
        "Instrumentation",
        "DG Set",
        "Transformers",
        "Cables",
        "Solar Power Plant",
        "TAS",
        "Panels",
        "MFM",
        "EM Locking System",
        "VTS",
        "Pail Filling Line",
        "Kettles",
        "HLVRM",
        "Fire Engine",
        "Air Compressors",
        "ETP",
        "Weighbridge",
        "Paging System",
        "Product Pumps",
        "Vapor Extraction System",
        "Hot Air Sealing Unit",
        "TFMS",
        "Electric Forklift",
        "DPT Equipment",
        "Gas Genset"
    ],

    "Civil & Works Contracts": [
        "Civil SOR",
        "Civil Works",
        "Security Services",
        "LPG Projects",
        "Marketing Services",
        "Biofuel Civil Works",
        "Used Cooking Oil",
        "Mobile Dispensers",
        "Major Civil Works",
        "Mastic Flooring",  # "Mastic Floor" SN → corpus: "mastic flooring works"
    "Floor Repair",
    "TV Area Civil",    # "Tv" SN → corpus: "civil m&r tv1/tv2"
    "Paint Primer",     # "Paint    Primer" SN (extra spaces stripped) → corpus: "paint primer"
    "Primer Supply",
    "UG Tank",          # "Ug" SN → corpus: "ug water tank"
    "UG Pipeline Civil",
    "Pit Testing",      # "Pit Test" SN → corpus: "earth pit testing jobs"
    "Earth Pit Testing",
    "Earth Pit Construction",
    ],

    "EPMC & Consultancy": [
        "Detailed Project Report",
        "Detailed Feasibility Report",
        "Environmental Studies",
        "Environmental Clearance",
        "Solar Consultancy",
        "Third Party Inspection",
        "Soil Investigation",
        "Market Survey",
        "LSTK Package",
        "Consultancy Services",
        "CVR"
    ],

    "LPG Equipment": [
        "LPG Cylinders",
        "Valves",
        "Regulators",
        "Safety Caps",
        "Tamper Evident Seals",
        "O-Rings",
        "Hot Repair",
        "Mechanical Works",
        "Civil Works",
        "Consultancy Services"
    ],

    "Logistics": [
        "Bulk POL Transportation",
        "Packed LPG Transportation",
        "Bulk ATF Transportation",
        "Bulk LPG Transportation",
        "Auto LPG Transportation",
        "CNG Transportation",
        "CBG Transportation",
        "Black Oil Transportation",
        "Bulk Bitumen Transportation",
        "Packed Bitumen Transportation",
        "Packed Lube Transportation",
        "Bulk Lube Transportation",
        "Rail Transportation",
        "Bitumen Rake Handling",
        "Base Oil Transportation",
        "Transformer Oil Transportation",
        "Material Transportation",
        "Lube Warehouse",
        "Secondary Transportation",
        "Ocean Freight",
        "Barges",
        "Aviation Refueling Manpower"
    ],

    "Packaging & Additives": [
        "Additives",
        "Packaging",
        "Raw Materials",
        "Handling Contracts",
        "Change PR Processing"
    ]
}
SUBCAT_TO_CATEGORY = {
    subcat: cat
    for cat, subcats in CATEGORY_MAP.items()
    for subcat in subcats
}
ALL_SUBCATS = list(SUBCAT_TO_CATEGORY.keys())

# ── Minimal stop words for matching (keep meaningful short tokens)
# ── Minimal stop words for matching (keep meaningful short tokens)
CATEGORY_STOP_WORDS = {
    'and', 'or', 'the', 'of', 'for', 'misc', 'general', 'other',
    # Action/process words — should not drive category mapping
    'repair', 'repairs', 'procurement', 'supply', 'supplies', 'purchase',
    'maintenance', 'installation', 'service', 'services', 'testing',
    'replacement', 'commissioning', 'fabrication', 'overhauling',
}

def _tokenize_for_match(text: str) -> set:
    words = re.sub(r'[^a-zA-Z\s]', ' ', str(text)).lower().split()
    return {w for w in words if w not in CATEGORY_STOP_WORDS and len(w) > 2}

# Pre-tokenize sub-category phrases once
_SUBCAT_TOKENS = {sc: _tokenize_for_match(sc) for sc in ALL_SUBCATS}

# ── Build cluster-level rich text: Standard Name + all raw titles joined
print("Building cluster title corpus for matching ...")
cluster_title_corpus = (
    df.groupby('Standard Name')[TITLE_COL]
      .apply(lambda x: ' '.join(x.astype(str).tolist()))
      .to_dict()
)
print(f"  Corpus built for {len(cluster_title_corpus):,} unique Standard Names")

def match_subcategory(standard_name: str, cluster_corpus: str = ""):
    """
    Two-stage matching:
      Stage 1: Try to match on Standard Name alone (high precision).
               If Standard Name gives a confident match, use it directly.
      Stage 2: If Stage 1 finds nothing, use corpus titles as a fallback
               (lower weight, only when Standard Name is very short / 1 token).
    This prevents corpus text from overriding what the Standard Name already says.
    """
    sn_tokens = _tokenize_for_match(standard_name)
    if not sn_tokens:
        return 'Unmapped', 'Unmapped'

    # ── Stage 1: Standard Name matching
    sn_best_subcat = None
    sn_best_key    = None

    for rank, subcat in enumerate(ALL_SUBCATS):
        sc_tokens = _SUBCAT_TOKENS[subcat]
        if not sc_tokens:
            continue
        matched_sn  = set()
        matched_sc  = set()
        if not sc_tokens.issubset(sn_tokens):
            continue
        matched_sc = sc_tokens
        matched_sn = sc_tokens & sn_tokens
        sn_coverage = len(matched_sn) / len(sn_tokens)
        sc_coverage = len(matched_sc) / len(sc_tokens)
        union_size  = len(sn_tokens | sc_tokens)
        jaccard     = len(matched_sn) / union_size if union_size else 0
        score = 0.5 * sn_coverage + 0.3 * sc_coverage + 0.2 * jaccard
        if score < 0.10:   # tighter threshold for SN-only match
            continue
        key = (round(score, 4), -rank)
        if sn_best_key is None or key > sn_best_key:
            sn_best_key    = key
            sn_best_subcat = subcat

    # If Standard Name gave a confident match, return it
    if sn_best_subcat is not None:
        return sn_best_subcat, SUBCAT_TO_CATEGORY[sn_best_subcat]

    # ── Stage 2: Corpus fallback — only when SN is a single token
    #    (e.g. Standard Name "Hose" alone won't match any subcat, but
    #     titles like "LPG HOSES" will reveal the right category)
    #    We do NOT use corpus when SN has 2+ tokens — it's precise enough.
    if len(sn_tokens) > 1:
        return 'Unmapped', 'Unmapped'

    corpus_tokens = _tokenize_for_match(cluster_corpus[:200])
    if not corpus_tokens:
        return 'Unmapped', 'Unmapped'

    corp_best_subcat = None
    corp_best_key    = None

    for rank, subcat in enumerate(ALL_SUBCATS):
        sc_tokens = _SUBCAT_TOKENS[subcat]
        if not sc_tokens:
            continue
        matched_corp = set()
        matched_sc   = set()
        for ct in corpus_tokens:
            for st in sc_tokens:
                if ct == st:
                    matched_corp.add(ct)
                    matched_sc.add(st)
        if not matched_corp:
            continue
        sc_coverage  = len(matched_sc) / len(sc_tokens)
        union_size   = len(corpus_tokens | sc_tokens)
        jaccard      = len(matched_corp) / union_size if union_size else 0
        # Require strong sub-category coverage from corpus
        # (prevents noise tokens from triggering a weak match)
        score = 0.6 * sc_coverage + 0.4 * jaccard
        if score < 0.20:   # stricter for corpus-only match
            continue
        key = (round(score, 4), -rank)
        if corp_best_key is None or key > corp_best_key:
            corp_best_key    = key
            corp_best_subcat = subcat

    if corp_best_subcat is None:
        return 'Unmapped', 'Unmapped'
    return corp_best_subcat, SUBCAT_TO_CATEGORY[corp_best_subcat]


print("Mapping clusters to Sub Category / Category ...")

def match_from_po_title(po_title: str):
    """Match Sub Category / Category directly from the raw PO title."""
    title_tokens = _tokenize_for_match(po_title)
    if not title_tokens:
        return 'Unmapped', 'Unmapped'

    best_subcat = None
    best_key    = None

    for rank, subcat in enumerate(ALL_SUBCATS):
        sc_tokens = _SUBCAT_TOKENS[subcat]
        if not sc_tokens:
            continue
        if not sc_tokens.issubset(title_tokens):
            continue
        key = (len(sc_tokens), -rank)
        if best_key is None or key > best_key:
            best_key    = key
            best_subcat = subcat

    if best_subcat is None:
        return 'Unmapped', 'Unmapped'
    return best_subcat, SUBCAT_TO_CATEGORY[best_subcat]

print("Mapping Sub Category / Category directly from PO titles ...")
results = df[TITLE_COL].apply(match_from_po_title)
df['Sub Category'] = results.apply(lambda x: x[0])
df['Category']     = results.apply(lambda x: x[1])

unmapped_count = (df['Category'] == 'Unmapped').sum()
mapped_count   = len(df) - unmapped_count
print(f"  Rows mapped     : {mapped_count:,} / {len(df):,}  ({mapped_count/len(df)*100:.1f}%)")
print(f"  Rows unmapped   : {unmapped_count:,}  ({unmapped_count/len(df)*100:.1f}%)")

if unmapped_count > 0:
    sample_unmapped = (
        df.loc[df['Category'] == 'Unmapped', TITLE_COL]
          .value_counts()
          .head(20)
    )
    print("\n  Top unmapped PO Titles (add keywords to CATEGORY_MAP to reduce further):")
    for nm, cnt in sample_unmapped.items():
        print(f"    [{cnt:>4}] {nm}")

unmapped_count = (df['Category'] == 'Unmapped').sum()
mapped_count   = len(df) - unmapped_count
print(f"  Rows mapped     : {mapped_count:,} / {len(df):,}  ({mapped_count/len(df)*100:.1f}%)")
print(f"  Rows unmapped   : {unmapped_count:,}  ({unmapped_count/len(df)*100:.1f}%)")

if unmapped_count > 0:
    sample_unmapped = (
        df.loc[df['Category'] == 'Unmapped', 'Standard Name']
          .value_counts()
          .head(20)
    )
    print("\n  Top unmapped Standard Names (add to CATEGORY_MAP to reduce further):")
    for nm, cnt in sample_unmapped.items():
        print(f"    [{cnt:>4}] {nm}")

df = df.sort_values(
    ['Category', 'Sub Category', 'Standard Name', 'Cluster ID'],
    ascending=[True, True, True, True]
).reset_index(drop=True)

print("\nCategory row distribution:")
for cat, cnt in df['Category'].value_counts().items():
    print(f"  {cat:<30} {cnt:>6} rows  ({cnt/len(df)*100:.1f}%)")

Building cluster title corpus for matching ...
  Corpus built for 5,594 unique Standard Names
Mapping clusters to Sub Category / Category ...
Mapping Sub Category / Category directly from PO titles ...
  Rows mapped     : 18,971 / 55,279  (34.3%)
  Rows unmapped   : 36,308  (65.7%)

  Top unmapped PO Titles (add keywords to CATEGORY_MAP to reduce further):
    [ 378] ESY2025-2026:C1.1
    [ 339] PROCUREMENT OF 14.2 KG CYLINDERS
    [ 323] ESY2025-2026:C1.2
    [ 317] ETHANOL CONTRACT ESY 2023-24 C1
    [ 278] ESY2025-2026:C1
    [ 276] ESY2024-2025:C1
    [ 219] PROCUREMENT OF 19 KG. CYLINDER
    [ 207] ESY2025-2026:C1.3
    [ 200] ESY2024-2025:C5.1
    [ 163] ESY2024-2025:C1.1
    [ 160] ESY2024-2025:C3.1
    [ 129] ESY2024-2025:C3
    [ 105] PROC OF 14.2 KG CYLINDERS
    [  91] 5 KG FTL AND DOM CYLINDER PROCUREMENT
    [  88] ESY2024-2025:C4
    [  84] ESY2025-2026:00
    [  80] ESY2025-2026:C1.2 SWZ
    [  72] ESY 2025-2026:C1.1 SWZ
    [  65] ETHANOL CONTRACT Q3 OF ESY 2023-24
    

## Step 11 · Build Excel Output

In [21]:
# Sheet 1: Clustered PO Titles  |  Sheet 2: Cluster Summary  |  Sheet 3: Stats Summary

HDR_FILL = PatternFill('solid', start_color='1F4E79')
ALT_FILL = PatternFill('solid', start_color='DCE6F1')
WHT_FILL = PatternFill('solid', start_color='FFFFFF')
HDR_FONT = Font(name='Arial', bold=True, color='FFFFFF', size=11)
DAT_FONT = Font(name='Arial', size=10)
CLR_FONT = Font(name='Arial', bold=True, size=10, color='1F4E79')
NM_FONT  = Font(name='Arial', italic=True, size=10, color='2E6DA4')
thin     = Side(style='thin', color='B0C4DE')
BORDER   = Border(left=thin, right=thin, top=thin, bottom=thin)
CENTER   = Alignment(horizontal='center', vertical='center')
WRAP     = Alignment(horizontal='left', vertical='center', wrap_text=True)
LEFT     = Alignment(horizontal='left', vertical='center')

output_cols = [TITLE_COL] + [c for c in EXTRA_COLS if c in df.columns]
HEADERS     = ['Cluster ID', 'Standard Name', 'Category', 'Sub Category'] + output_cols

wb = Workbook()

# --- Sheet 1: Clustered PO Titles ---
ws = wb.active
ws.title = "Clustered PO Titles"
for c, h in enumerate(HEADERS, 1):
    cell = ws.cell(row=1, column=c, value=h)
    cell.font = HDR_FONT; cell.fill = HDR_FILL
    cell.border = BORDER; cell.alignment = CENTER
ws.row_dimensions[1].height = 22

prev_cid, alt = None, False
for r_idx, row in df.iterrows():
    excel_row = r_idx + 2
    cid = row['Cluster ID']
    if cid != prev_cid:
        alt = not alt
        prev_cid = cid
    fill = ALT_FILL if alt else WHT_FILL
    for c, col in enumerate(HEADERS, 1):
        val  = row.get(col, '')
        cell = ws.cell(row=excel_row, column=c, value=val)
        cell.border = BORDER; cell.fill = fill
        if col == 'Cluster ID':
            cell.font = CLR_FONT; cell.alignment = CENTER
        elif col == 'Standard Name':
            cell.font = NM_FONT; cell.alignment = WRAP
        elif col in ('Category', 'Sub Category'):
            cell.font = DAT_FONT; cell.alignment = WRAP
        elif col == TITLE_COL:
            cell.font = DAT_FONT; cell.alignment = WRAP
        else:
            cell.font = DAT_FONT; cell.alignment = LEFT
    ws.row_dimensions[excel_row].height = 18

col_widths = [12, 35, 25, 30, 55] + [22] * (len(output_cols) - 1)
for i, w in enumerate(col_widths, 1):
    ws.column_dimensions[get_column_letter(i)].width = w
ws.freeze_panes = 'A2'
ws.auto_filter.ref = f"A1:{get_column_letter(len(HEADERS))}1"
wb.save(OUTPUT_FILE)
print(f"\nSaved -> {OUTPUT_FILE}")


Saved -> Clustered_Output_final_testing_2.xlsx


In [22]:
# Check unmapped count in output Excel
df_check = pd.read_excel(OUTPUT_FILE)
unmapped = (df_check['Category'] == 'Unmapped').sum()
total    = len(df_check)
mapped   = total - unmapped

print(f"Total rows   : {total:,}")
print(f"Mapped       : {mapped:,}  ({mapped/total*100:.1f}%)")
print(f"Unmapped     : {unmapped:,}  ({unmapped/total*100:.1f}%)")


Total rows   : 55,279
Mapped       : 18,971  (34.3%)
Unmapped     : 36,308  (65.7%)


In [23]:
# ── Second-pass category map for unmapped rows only
CATEGORY_MAP_2 = ({

    "Instrumentation": [
        "PLC",
        "DCS",
        "Analyzers",
        "Control Valves",
        "ON/OFF Valves",
        "Valve Spares",
        "Level Gauges",
        "Level Transmitters",
        "Pressure Gauges",
        "Transmitters",
        "Radar Tank Gauging System",
        "Gas Chromatographs",
        "Gas Detectors",
        "Flame Scanner",
        "Corrosion Monitors",
        "Public Address System",
        "HVAC Shelter",
        "Instrument Canopy",
        "Junction Boxes",
        "Instrument Fittings",
        "Instrumentation Cables",
        "Communication Cables",
        "Power Cables",
        "Signal Cables",
        "Cable Laying",
        "Ambient Monitoring System",
        "Stack Monitoring System",
        "Rim Seal System",
        "PSA Valve Spares",
        "SPM Spares",
        "Calibration Gas Cylinders",
        "Bently Spares",
        "CCTV",
        "Fire Alarm System",
        "Clean Agent System",
        "Flameproof AC",
        "Flameproof Painting",
        "HMI",
        "AC Maintenance",
        "Refrigeration Maintenance",
        "Control Valve Overhauling",
        "Instrumentation Maintenance",
        "Instrumentation AMC",
        "Instrumentation"
    ],

    "IT - Refinery": [
        "Printers",
        "Monitors",
        "Hardware",
        "Software",
        "IT AMC",
        "ITES Services"
    ],

    "HR & Administration": [
        "Medicines",
        "Medical Devices",
        "Doctor Services",
        "Access Control Room",
        "Telephone Services",
        "Stationery",
        "Gate Security",
        "Bus Services",
        "Taxi Services",
        "Canteen Services",
        "Cafeteria Services",
        "Fire Retardant Coverall",
        "Courier Services",
        "Mail Services",
        "Provisions",
        "Laundry Services",
        "Training Services",
        "Certification Services",
        "Third Party Services",
        "Equipment CAMC",
        "Printing Services",
        "Event Management",
        "Audit Services",
        "Documentation Services",
        "Manpower Services",
        "Subscription Services",
        "CHA Services"
    ],

    "Projects - Refinery": [
        "Major Projects",
        "Minor Projects"
    ],

    "Materials & Warehouse": [
        "Outsourced Manpower",
        "Logistics Services",
        "CHA Services",
        "Expediting Services",
        "Housekeeping Services",
        "Compactor AMC",
        "Material Handling",
        "Packing Waste Disposal",
        "Document Scanning",
        "Warehouse Services"
    ],

    "Fire & Safety": [
        "PPE",
        "Fire Fighting Equipment AMC",
        "Hydrotesting",
        "Fire Equipment Refilling",
        "Outsourced Manpower",
        "Safety Services",
        "Third Party Services",
        "Training",
        "External Audit",
        "Certification Services",
        "Mahalakshmi Theatre Maintenance"
    ],

    "Laboratory": [
        "Lab Equipment",
        "Lab Spares",
        "Lab Maintenance",
        "Lab Calibration",
        "Lab Services",
        "Certified Reference Materials",
        "Proficiency Testing",
        "Third Party Services",
        "External Audit",
        "Certification Services",
        "Outsourced Manpower",
        "Laboratory",
        "Oil Testing",              # "Test" SN → corpus: "tr oil testing"
    "Air Quality Testing",
    "Bitumen Testing",
    "Tank Calibration",         # "Calibration Tanks" SN match
    "Glassware Calibration",
    "W&M Calibration",
    "Gas Detector Calibration",
    "Auto Titrator",            # "Titrator" SN → corpus: "auto titrator"
    "Titrator Spares",
    "Sample Management",        # "Sample Container" SN → corpus: "lab sample management"
    "RLA Sample",
    "GC Machine",               # "Gc" SN → corpus: "gc machine calibration"
    "GC Calibration Gas",
    "CRM GC",
    "Distillation Equipment",   # "Distillation" SN → corpus: "auto distillation"
    "Auto Distillation",
    "CFR Engine",               # "Cfr Engine" SN match
    "ICP Analyser",             # "Icp" SN → corpus: "icp oes", "icp analyzer"
    "UTG Survey",               # "Utg" SN match
    "Ultrasonic Thickness",     # "Ultrasonic Thickness" SN match
    "Lab Analyst",              # "Analyst" SN → corpus: "lab analyst"
    "Lab Infrastructure",       # "Medical Infrastructure" SN → corpus: "lab infrastructure"
    "Lab Assistance",           # "Stand" SN → corpus: "standing pr for lab assistance"
    "Conductivity Meter",
    "Pour Point",               # "Pour" SN → corpus: "auto pour point analyser"
    "Test Bath",                # "Test Bath" SN match
    "Weighing Balance",         # "Weigh" → corpus: "weighing balance for haldia qc lab"
    "Flow Meter Calibration",
    "Statutory Testing",
    "Tank Inspection",
    "Thickness Survey",
    "Environmental Testing",
    ],

    "Technical Process / CES / MES": [
        "Engineering Consultancy",
        "Documentation",
        "Drafting Services",
        "Liaisoning Services",
        "Third Party Services",
        "External Audit",
        "Certification Services",
        "Liaisoning Stamping",      # "Liasoning"/"Noc Liasoning" SN match
    "NOC Liaisoning",
    "DM NOC",
    "PESO Liaisoning",          # "Peso Liasoning" SN match
    "CNG Endorsement",
    "Bitumen Supervision",      # "Supervision" SN → corpus: "bitumen ship supervision"
    "TT Supervision",
    "Feasibility Study",        # "Study" SN → corpus: "feasibility study"
    "EIA Study",
    "DFR PMC",
    "Engineering Services",     # "Engg" SN → corpus: "rate contract for engg services"
    "Detail Engineering",
    "EPMC Services",            # "Epmc" SN match
    "Insulation Joint",         # "Track" → corpus: "insulation joint rly track"
    "Digital Water Meter",      # "Digital" SN → corpus: "digital water meters"
    "Digital Equipment",
    "Digital Display",          # "Display" SN → corpus: "digital display system"
    "CEG Solution",             # "Solution" SN → corpus: "development of ceg solution"
    "IT Solution",
    "SCADA Upgrade",            # "Scada" SN match
    "RTO Model",                # "Model" SN → corpus: "rto model development"
    "3D Model",
    "Composite Survey",         # "Composite" SN → corpus: "encon composite survey"
    "ENCON Survey",
    "Valuation Services",       # "Valuation" SN → corpus: "valuation of land"
    "Land Valuation",
    "IBR RLA",                  # "Ibr" SN → corpus: "rla of ibr equipment"
    "IBR Equipment SO",
    "Surveillance Contract",
    "CDCMS Services",
    "CES Civil",                # "Ces Civil" SN match
    "SAP Technical",            # "Sap" SN → corpus: "sap technical manpower"
    "CHOS Integration",
    ],
    "EPMC": [
    "Architectural Services",
    "QRA Study",                # "Qra Hazop" → corpus
    "HAZOP QRA",
    "HIRA Study",
    "EIA Clearance",
    "DFR Preparation",
    "PMC Project",
    "Survey Design",
    "Inspection Agency",        # "Inspection Tank" → corpus: "inspection agency"
    "TPI Projects",
    "Feasibility",
    "Advisory Services",        # "Report" SN → corpus: "advisory services"
    "Assurance Report",
    "BRSR Report",
    ],
    "Operations": [
        "Oily Sludge Cleaning",
        "Catalyst Disposal",
        "Hazardous Waste Disposal",
        "Software Procurement",
        "Software Subscription",
        "Unit O&M",
        "ISPRL Operations",
        "Launch Services",
        "Shipping Services",
        "Safety Manpower Operations",
    "Gantry Operations",
    "TT Assistance",
    "TT Supervision",
    "Tank Farm Operations",
    "Terminal Operations Manpower",
    "Operational Assistance",
    "Lube COD Handling",
    "Handling Contract",
    "Loading Unloading",
    "Rake Loading",
    "STS Operation",            # "Sts" SN → corpus: "sts operation"
    "Bunker Survey",
    "Marine Survey",            # "Vessel Survey"/"Tanker" SN → corpus
    "LPG Vessel Survey",
    "Quantum Survey",
    "AC Interference Survey",   # "Ac Interference" SN match
    "Pipeline Patrolling",      # "Surveillance" → corpus: "patrolling surveillance"
    "Pipeline Pigging",
    "CNG Transportation",       # "Transprtation Hcv" SN → corpus
    "CNG Transport HCV",
    "TPT Lube",                 # "Tpt Ex"/"Tpt" SN match
    "Black Oil Transport",
    "Custom Clearance",
    "Manpower Lube Handling",   # "Lube Handling" SN match
    "Lube Handling",
    "Loading Manpower",         # "Manpower" SN → corpus: "manpower services o&m"
    "O&M Manpower",
    "Bottling Operation",       # "Bottle" SN → corpus: "lpg bottling operation"
    "Private Bottling",
    "CNG Station",              # "Ddd"/"Tank" SN → corpus: "cng and tank installation"
    "CNG Tank Installation",
    "EVC Installation",         # "Evc" SN → corpus: "evc installation works"
    "EVCS Works",
    "FAME EVCS",                # "Fame" SN match
    "Net Metering",             # "Net Zero" SN → corpus: "net metering installation"
    "Net Zero Certification",
    "ESY Contract",             # "Cy"/"Qty"/"Addn" SN → corpus: "esy" contracts
    "ESY Ethanol",
    "Ethanol Allocation",       # "Bd Allocation" analog
    "CBG Operations",
    "CGD Operations",
    "MDPE LMC",                 # "Mdpe Lmc" SN match
    "LMC Works",
    "Route Survey CGD",         # "Detail Route" SN match
    "CGD Survey",
    "Track Maintenance",        # "Track" SN → corpus: "railway track maintenance"
    "Railway Track",
    "Jeep Hire",                # "Jeep" SN → corpus: "office jeep hiring"
    "Manpower ELE",             # "Ele"/"Opns"/"Opn" SN → corpus: "ele manpower service"
    "Operations Manpower",
    "Upkeep Manpower",
    "General Maintenance Manpower", # "General"/"Maintenace" SN match
    "Plant Maintenance Manpower",
    "Manpower Comco",           # "Manpower Comco" SN match
    "Manpower RMC",             # "Rmc" SN → corpus: "manpower supply comco rmc"
    "NPV Services",             # "Npv" SN → corpus: "hiring of npv"
    "M&R Works",                # "Jobs"/"Plains"/"Hills"/"Civil Kol" SN → corpus
    "M&R Hills",
    "M&R Plains",
    "M&R Sales Area",
    "Sales Area Maintenance",
    "ELE M&R",                  # "Electrcial"/"Elect" SN → corpus: "electrcial m&r works"
    "Electrical M&R",
    "Ethanol",          # "Ethanol Rajasthan", "Ethanol", "Ethanol Procur" all contain "ethanol"
    "Biodiesel",        # "Biodiesel", "Bio Diesel", "Bd Allocation" → corpus has "biodiesel"
    "Biofuel",          # "Biofuel" std names
    "Crude Survey",     # "Crude" → corpus has "crude vessel survey"
    "Residue",          # std name "Residue"
    "Liquid Propane",   # std name "Liquid Propane"
    "Green Hydrogen",   # "Green" → corpus has "green h2", "green hydrogen"
    "Pipeline Survey",  # "Pipeline" (326 rows) → corpus has "pipeline route survey"
    "Pipeline Inspection",
    "Fit For Road",     # "Road" (152 rows) → corpus has "fit for road"
    "Transportation Lube",  # std name exact match
    "Bitumen Supervision",  # "Supervision" → corpus has "bitumen ship supervision"
    "Vessel Survey",    # "Survey Biomass"/"Inspection Tank" → corpus has "vessel survey"
    "TPT",
        
        
    ],

    "Miscellaneous": [
        "ISPRL Services",
        "Supplies",
        "Services"
    ],


    "Chemicals - Refinery": [
        "Coagulants",
        "Poly Aluminum Chloride",
        "Rock Salt",
        "Ethyl Mercaptan",
        "DMDS",
        "Perchloroethylene",
        "Antioxidants",
        "BHT",
        "Alum",
        "Citric Acid",
        "Sodium Hypochlorite",
        "Sodium Bisulphite",
        "Sodium Metabisulphite",
        "Dewatering Polyelectrolyte",
        "MBR Antifoaming Agent",
        "Deoiling Polyelectrolyte",
        "Antiscalant",
        "Activated Clay",
        "Caustic Lye",
        "Sodium Hydroxide",
        "Hydrochloric Acid",
        "Liquid Propane",
        "Liquor Ammonia",
        "Liquid Nitrogen",
        "Methanol",
        "Hydrazine Hydrate",
        "Hydrogen Peroxide",
        "MDEA",
        "Shift Converter",
        "HGU Adsorbent",
        "Soda Ash",
        "Morpholine",
        "Corrosion Inhibitor",
        "Demulsifier",
        "DM Plant Resins",
        "Activated Alumina",
        "Granular Activated Carbon",
        "Inert Ceramic Balls",
        "Glass Beads",
        "Activated Charcoal",
        "Glycol",
        "Chloride Adsorbent",
        "AFFF",
        "Defoamer",
        "R&D Chemicals",
        "SWRO Chemicals",
        "Cooling Tower Chemicals",
        "BCW Chemicals",
        "IETP Chemicals",
        "Laboratory Chemicals",
        "Cooking Oil",
        "Filter Clay",
        "Filter Media",
        "Neutralizing Amine",
        "Chemicals"
    ],

    "Solvents": [
        "N-Methyl-2-Pyrrolidone",
        "Industrial Solvents",
        "Specialty Solvents"
    ],

    "Civil - Refinery": [
        "Building Construction",
        "Road Construction",
        "Trench Construction",
        "Civil Repairs",
        "Structural Fabrication",
        "Retrofitting",
        "Rehabilitation",
        "Fireproofing",
        "Painting",
        "Waterproofing",
        "Roof Repair",
        "Housekeeping",
        "Janitorial Services",
        "Road Cleaning",
        "Grass Cutting",
        "Dredging",
        "Tank Pad Repair",
        "Boundary Wall Repair",
        "Drain Cleaning",
        "Pipe Alley Cleaning",
        "SPM Civil Works",
        "Garbage Disposal",
        "Debris Disposal",
        "Oil Catcher Desilting",
        "Bay Cleaning",
        "Dewatering Pump Services",
        "OWS Cleaning",
        "Pest Control",
        "Landscaping",
        "Garden Maintenance",
        "Tree Trimming",
        "Steel Structure AMC",
        "RCC Consultancy",
        "Plumbing",
        "Sanitary Services",
        "Carpentry",
        "Chair Repair",
        "Sofa Repair",
        "Signage",
        "Rolling Shutters",
        "Housing Colony Maintenance",
        "Drum Cleaning",
        "Animal Control",
        "Hay Filters",
        "Civil Works"
    ],

    "Electrical - Refinery": [
        "Electrical Equipment",
        "Motors",
        "Generators",
        "Transformers",
        "Panels",
        "UPS",
        "Public Address System",
        "HT Motors",
        "LT Motors",
        "Switchgear",
        "Battery Maintenance",
        "UPS Maintenance",
        "MOV Actuators",
        "PA System Services",
        "NIFPS",
        "HVAC",
        "Air Conditioning",
        "Illumination",
        "Substation O&M",
        "Load Sharing",
        "MSS Maintenance",
        "LV Switchgear",
        "FRLS Cables",
        "Cable Ducts",
        "Power Modules",
        "Substation Housekeeping",
        "Zonal Electrical Maintenance",
        "Transformer Maintenance",
        "Elevators",
        "Lighting Towers",
        "High Mast",
        "GIS",
        "VFD",
        "Motor Control Panels",
        "Heaters",
        "SCADA",
        "APM System",
        "Electrical HMI",
        "Cathodic Protection",
        "GTG Spares",
        "Preventive Maintenance",
        "Testing During TA",
        "SPM Electrical Spares",
        "Open Access Power",
        "REC",
        "Heat Tracing",
        "Battery Chargers",
        "Desalter Spares",
        "Ignitor Spares",
        "Capacitor Banks",
        "Fire Siren System",
        "HVAC Gas Cylinders",
        "Cable Lugs",
        "Cable Glands",
        "Electrical Consumables",
        "Electrical"
    ],
    "Mechanical - Refinery Materials": [
        "Valves",
        "Pipes",
        "Pipe Fittings",
        "Flanges",
        "Structural Plates",
        "Structural Steel",
        "Gaskets",
        "Nuts and Bolts",
        "Exchanger Components",
        "Tower Internals",
        "Filter Elements",
        "Skids",
        "Burner Spares",
        "Refractory",
        "Pressure Vessels",
        "Vessel Internals",
        "Heat Exchangers",
        "Towers",
        "Reactors",
        "Furnace Tubes",
        "Pump",
        "Pump Spares",
        "Air Fin Coolers",
        "APH Tubes",
        "Bellows",
        "Couplings",
        "Coupling Spares",
        "Radial Shaft Seals",
        "Belts",
        "Pulleys",
        "Governors",
        "Gas Turbine Components",
        "OEM Spares",
        "Compressors",
        "Blowers",
        "Mechanical Seals",
        "Mechanical Seal Spares",
        "Bearings",
        "Lube Oil",
        "O-Rings",
        "Packing",
        "Strainers",
        "Filters",
        "Gear Boxes",
        "Turbines",
        "Cooling Tower Components",
        "Automobile Equipment",
        "Lifting Equipment",
        "Fire Fighting Equipment",
        "Level Gauges",
        "SPM Equipment",
        "Mechanical Consumables",
        "Industrial Lubricants",
        "Mechanical Items"
    ],

    "Mechanical - Refinery Contracts": [
        "Mechanical Static Contract",
        "Mechanical Rotary Contract",
        "Mechanical Workshop Contract",
        "Mechanical Garage Contract",
        "Mechanical Tankage Contract",
        "Mechanical Painting Contract",
        "Mechanical Insulation Contract",
        "Mechanical Turnaround Contract",
        "Catalyst Loading Contract",
        "Chemical Loading Contract",
        "Crane Services",
        "EOT Services",
        "HOT Services",
        "Scaffolding Contract",
        "Lubrication Contract",
        "Preventive Maintenance",
        "Pump Supervisory Services",
        "Compressor Supervisory Services",
        "Condition Monitoring",
        "Valve Testing",
        "Valve Servicing",
        "Fire Fighting Equipment AMC",
        "Equipment Hiring",
        "Transportation Services",
        "Online Leak Arresting",
        "Hot Tapping",
        "Gas Turbine AMC",
        "Inspection Services",
        "Underwater Inspection",
        "Crane Operation"
    ],

    "Chemical Additives - Refinery": [
        "Antistatic Additive",
        "Nickel Passivator",
        "CO Combustion Promoter",
        "Dewaxing Aid",
        "HSD Lubricity Additive",
        "Multifunctional Additives",
        "Naphtha Lubricity Additive",
        "ZSM-5 Additive",
        "Orange Dye",
        "Antifoaming Agent",
        "Defoamer",
        "Pour Point Depressant",
        "Octane Booster",
        "Additives"
    ],

    "Catalysts - Refinery": [
        "FCCU Catalyst",
        "ECAT",
        "Merox Catalyst",
        "FB Reagent",
        "Thoxcat ES",
        "Sulphur Guard Bed Catalyst",
        "CCR Platforming Catalyst",
        "Diesel Isotherming Catalyst",
        "DHT Catalyst",
        "Prime-G HDS Catalyst",
        "Prime-G SHU Catalyst",
        "Bensat Catalyst",
        "Prime G LD 412 R Catalyst",
        "LOUP Catalyst",
        "HGU PDS Catalyst",
        "ISOM Catalyst",
        "COMO Catalyst",
        "Hydrofiner Catalyst",
        "Sulphur & Chlorine Absorber Catalyst",
        "VRMP Catalyst",
        "HGU Reformer Catalyst",
        "FCHCU Catalyst",
        "DHDS Catalyst",
        "DIU Catalyst",
        "RUF EB Catalyst",
        "RUF Hydrotreating Catalyst",
        "RUF Fixed Bed Catalyst",
        "Hydroprocessing Catalyst",
        "Hydrotreating Catalyst",
        "Hydrocracking Catalyst",
        "Catalytic Reforming Catalyst",
        "Isomerization Catalyst",
        "Alkylation Catalyst",
        "Catalysts"
    ],
    "Operations": [
        # Fuels & feedstock — covers Ethanol (~4600 rows), Biodiesel (~180 rows)
        "Ethanol",
        "Biodiesel",
        "Biofuel",
        "Green Hydrogen",
        "Hydrogen",
        "Natural Gas",
        "CNG Supply",
        "LNG Supply",
        "Petroleum Products",
        "Crude Oil",
        "Naphtha",
        "Residue",
        "Slop Oil",
        "Waxy Distillate",
        "Transformer Oil Supply",
        "Base Oil Supply",
        "Furnace Oil",
        "HSD Supply",
        # Pipeline operations — covers Pipeline cluster (~245 rows)
        "Pipeline Survey",
        "Pipeline Route Survey",
        "Pipeline Thickness Survey",
        "ROW Survey",
        "IP Survey",
        # Fit-for-road / terminal operations
        "Fit For Road",
        "Terminal Operations",
    ],
 
    "LPG Equipment": [
        # Cylinder cluster — ~977 rows
        "Cylinder Handling",
        "LPG Cylinder Handling",
        "SCBA Cylinder",
        "Cylinder Filling",
        "Cylinder Maintenance",
        "LPG Bottling",
        "Safety Caps Supply",
        "SC Caps",
        "Valve Caps",
        # Actuators — ~88 rows
        "Actuator Spares",
        "Electric Actuator",
        "Rotork Actuator",
        "RoV Actuator",
        # Conveyor — ~96 rows (LPG bottling)
        "Conveyor Chain",
        "Bottling Conveyor",
        "Conveyor Spares",
        # Liaisoning / PESO — ~164 + ~116 rows
        "PESO Approval",
        "CNG PESO",
        "Liaisoning",
        "Stamping Job",
        "CNG Endorsement",
        "Cylinder Handling",    # "Cylinder" SN → corpus: "cylinder handling"
    "LPG Cylinder",         # Stage 2 corpus match
    "SCBA Cylinder",        # Stage 2 corpus match
    "PESO",                 # "Peso" SN → single token, matches directly
    "SC Caps",              # "Sc" SN → corpus: "sc caps"
    "Safety Caps",          # Stage 2 fallback
    "Actuator Spares",      # "Actuator" SN → corpus: "actuator oil seal"
    "MOV Actuator",
    "GMS Sensors",          # "Gm" SN → corpus: "gms sensors", "gms system"
    "GMS System",
    "OVCM",                 # "Ovcm" SN → single token match
    "OVCM Overhauling",
    "Sumo Cylinder",        # std name exact match
    "Statutory Testing",    # "Statutory Test" → corpus: "statutory testing of cyls"
    "Degassing Unit",       # "Degas" SN → corpus: "degassing unit"
    "Combo Valve",          # std name match
    "SC Valve",             # std name match
    "ROV",                  # "Rov" SN → corpus: "rosov", "rov pov"
    "ROSOV",
    "Breakaway Coupler",    # std name match
    "DBBV",                 # "Dbbv" SN exact
    "Conveyor Chain",       # "Conveyor" SN → corpus: "conveyor chain lpg bottling"
    "Bottling Conveyor",
    "Liaisoning",           # "Liasoning" SN → corpus: "liasoning stamping job"
    "Stamping Job",
    "TES Seals",            # "Tes" SN → corpus: "tes seals in lpg plants"
    "DGCC Books",           # "Dgcc Book" SN → corpus: "dgcc books"
    "Valve Salvage",        # "Valve Salvage" SN match
    "Cap Salvage",
    "Retest Cylinder",      # "Retest Cylinder" SN match
    "Evacuation Unit",      # "Evacuation" SN → corpus: "evacuation unit for 425 kg"
    "Stenciling",           # "Stencil" SN → corpus: "1906 stenciling cylinders"
    "LPG Pump",             # "Pump" SN → corpus: "repair of lpg pump"
    "LPG Pump Spares",
    "Pump Spares",
    "Tank Wagon",           # "Tank Wagon" SN match
    "CNG Cascade",          # "Cascade" SN → corpus: "cng storage mobile cascade"
    "Sprinkler Line",       # "Sprinkler" SN → corpus: "sprinkler line replacement"
    ],
 
    "Electrical & Instrumentation": [
        # DG Set — ~180 rows
        "DG Set Servicing",
        "DG Oil Servicing",
        "DG Set Overhauling",
        "DG Set Maintenance",
        "DG Set Spares",
        # Air Compressor — ~163 + 96 rows
        "Air Compressor Maintenance",
        "Air Compressor Repair",
        "Air Compressor Spares",
        "KPCL Compressor",
        "Compressor B-Check",
        # AC / HVAC — ~93 rows
        "Air Conditioner Procurement",
        "Air Cooled AC",
        "Package AC",
        "AC Repair",
        # Earthpit — ~86 rows
        "Earthpit Testing",
        "Earth Pit Testing",
        "Earthing Testing",
        # Cables — ~93 rows
        "HT Cable",
        "LT Cable",
        "Cable Trench",
        "Cable End Termination",
        # VHF / Communication — ~85 rows
        "VHF Sets",
        "VHF Walkie Talkie",
        "VHF Repeater",
        "Walkie Talkie",
        # Analysers — ~81 rows
        "Air Demand Analyser",
        "Wear Analyser",
        "Nitrogen Analyser",
        "Gas Analyser",
        # FLP — ~85 rows
        "FLP Lights",
        "FLP Enclosures",
        "FLP Siren",
        "Flameproof Lights",
        # Linear lights — ~88 rows
        "Linear Light",
        "Linear Filling Machine",
        # Smart Pylons / Meters — ~110 rows
        "Smart Pylon",
        "Smart Water Meter",
        # Monitoring equipment — ~71 rows
        "Monitoring Equipment",
        "Water Monitor",
        "Ground Water Monitoring",
        # Hydrant — ~101 rows
        "Hydrant Dispensers",
        "Hydrant Line Survey",
        "Hydrant Items",
        "DG Set",               # "Dg" SN → corpus: "dg set", "dg oil servicing"
    "DG Oil",
    "Air Compressor",       # std name exact 2-token match
    "Compressor Spares",    # "Compressor" SN → corpus: "compressor spares", "air compressor"
    "Atlas Copco",          # "Atla Copco" SN → corpus: "atlas copco air compressor"
    "IR Compressor",        # "Ir Compressor" SN match
    "Air Conditioner",      # "Ac" SN → corpus: "air conditioner", "air cooled ac"
    "Split AC",             # "Split Ac" SN match
    "Earthpit Testing",     # "Earthpit Test" SN → corpus: "earthpit testing"
    "Earthpit Construction",# "Earth Pit"/"Earthpits"/"Earthpit" → corpus
    "Earthpit",             # "Earthpit" SN single token
    "Earthing System",      # "Earth" SN → corpus: "earthing and lightning"
    "Earthing Audit",
    "Cable Trench",         # "Cable" SN → corpus: "cable trench", "ht cable"
    "HT Cable",
    "VHF Sets",             # "Vhf" SN → corpus: "vhf sets", "vhf walkie talkie"
    "VHF Repeater",
    "Analyser Spares",      # "Analyser"/"Analyzer" SN → corpus
    "Gas Analyser",
    "Crude Analyzer",
    "Breath Analyzer",
    "FLP Lights",           # "Flp" SN → corpus: "flp lights"
    "FLP Enclosures",
    "Linear Light",         # "Linear" SN → corpus: "linear light installation"
    "Linear Filling Machine",
    "Smart Pylon",          # "Smart Pylon" SN → corpus: "new smart pylon"
    "Smart Water Meter",
    "Condition Monitoring", # "Monitor" SN → corpus: "condition monitoring services"
    "Monitoring Equipment",
    "PLC",                  # "Plc" SN → corpus: "plc upgradation", "plc procurement"
    "PLC Upgradation",
    "VFD",                  # "Vfd" SN → corpus: "vfd spares", "vfd maintenance"
    "VFD Spares",
    "PCU",                  # "Pcu" SN → corpus: "pcu maintenance", "sitc of pcu"
    "Battery Bank",         # "Battery"/"Battery Bank" SN → corpus: "battery bank for ups"
    "Battery Charger",
    "UPS Battery",
    "Transmitter",          # "Transmitter" SN match
    "Pressure Transmitter",
    "Radar Level",
    "Flowmeter",            # "Flowmeter"/"Flow" SN → corpus: "flow meter"
    "Flow Meter",
    "Digital Flowmeter",
    "Servo Gauge",          # "Servo Gauge" SN match
    "Level Gauge",          # "Gauge" SN → corpus: "level gauge", "pressure gauge"
    "Pressure Gauge",
    "ATG Probe",            # "Atg Probe" SN match
    "ATG System",
    "HCD Detector",         # "Hcd"/"Path Hcd" SN → corpus: "hcd gas detector"
    "Gas Detector",         # "Gas" SN → corpus: "multi gas detector"
    "Hydrant Dispensers",   # "Hydrant" SN → corpus: "hydrant dispensers fabrication"
    "Hydrant Line",
    "CNG Dispenser",        # "Dispenser" SN → corpus: "car auto cng dispensers"
    "DU Dispenser",
    "VCB",                  # "Vcb" SN → corpus: "11 kv vcb", "vcb room"
    "SCADA",                # "Scada" SN → corpus: "scada system replacement"
    "CVRS System",          # "Cvrs" SN match
    "PIDS System",          # "Pids" SN match
    "Alarm System",         # "Alarm" SN → corpus: "fire alarm system"
    "Smoke Detection",      # "Smoke" SN → corpus: "smoke detection system"
    "Surge Protection",     # "Device" SN → corpus: "surge protection device"
    "Access Control",       # "Access" SN → corpus: "access control system"
    "Door Frame Detector",  # "Door Frame" SN match
    "Baggage Scanner",      # "Baggage Scanner" SN match
    "DFMD",                 # "Dfmd" SN match
    "Rectifier",            # "Rectification" SN → corpus: "lighting rectification"
    "EV Charger",           # "Ev" SN → corpus: "ev charger, ev fast charger"
    "EVCS",                 # "Evc"/"Evfs" SN → corpus: "evc installation"
    "Pelmet Lighting",      # "Pelmet" SN match
    "Additional Lighting",  # "Additional Lighting" SN match
    "Illumination",         # "Illumination" SN match
    "Tyre Inflator",        # "Tyre Inflator" SN match
    "ORPAK Automation",     # "Orpak" SN match
    "Automation Spares",    # "Automation Spare"/"Automation Spares" SN match
    "Automation Slave",     # "Slave" SN → corpus: "automation slave"
    "Relcon Automation",    # "Relcon" SN match
    "DV System",            # "Dv" SN → corpus: "dv system", "dv indication panel"
    "Phase Automation",     # "Iii Electrification" SN → corpus: "phase upgradation"
    "PULSAR Upgrade",       # "Pulsar" SN → corpus: "pulsar upgradation"
    "AV System",            # "Av" SN → corpus: "av system", "av equipment"
    "VSAT",                 # "Vsat"/"Vsat Bandwidth" SN match
    "Telecom System",       # "Telecom" SN → corpus: "telecom system"
    "PA System",            # "Pa" SN → corpus: "pa system for main gates"
    "Network Equipment",    # "Network" SN → corpus: "network equipment"
    "Camera System",        # "Camera" SN → corpus: "thermal imaging camera"
    "CCTV Camera",
    "Proximity Sensor",     # "Sensor" SN → corpus: "proximity sensors"
    "Limit Switch",         # "Limit" SN → corpus: "limit switch"
    "Thermocouple",         # "Thermocouple" SN match
    "Conductivity Meter",   # "Conductivity" SN match
    "Earth Resistance Tester", # "Tester" SN → corpus: "earth resistance tester"
    "Weighing Scale",       # "Weigh"/"Weigh Bridge" SN → corpus: "weighing scale"
    "Weighbridge",
    "Load Cell",            # "Cell" SN → corpus: "load cell replacement"
    "High Mast",            # "Highmast" SN match
    "Yard Light",           # "Yard" SN → corpus: "yard light works"
    "FLP Pump",             # "Flp" corpus: "oil skimmer flp pump"
    "Magnetic Level Gauge", # "Magnetic" SN → corpus: "magnetic level gauge"
    "Siemens PLC",          # "Siemens" SN → corpus: "siemens plcs"
    "ABB VFD",              # "Abb" SN → corpus: "abb vfd maintenance"
    "Fisher Valve",         # "Fisher Valve" SN match
    "UTG Services",         # "Utg" SN → corpus: "emat utg"
    ],
 
    "Mechanical": [
        # Hoses — ~123 rows
        "Steam Hoses",
        "Air Hoses",
        "Hose Rate Contract",
        # Compressor spares — ~96 rows (non-LPG)
        "Compressor Overhaul",
        "Compressor Spares",
        # GMS sensors — ~72 rows
        "GMS Sensors",
        "GMS Spares",
        "Gas Monitoring Sensors",
        # Mechanical assistance — ~244 rows (generic std name)
        "Mechanical Assistance",
        "Mechanical Maintenance Assistance",
        # Spares (generic) — ~94 rows
        "MOV Spares",
        "DU Spares",
        "General Spares",
        # Shifting jobs — ~83 rows
        "Equipment Shifting",
        "Dismantling Shifting",
        "Material Shifting",
        "Mechanical Assistance",    # "Mechanical" SN → corpus: "mechanical assistance"
    "Mechanical Maintenance",
    "Mechanical Manpower",
    "Air Compressor Spares",    # "Compressor" SN → corpus: "air compressor spares"
    "Compressor Overhaul",
    "KPCL Compressor",
    "Steam Hose",               # "Hose" SN → corpus: "steam hoses", "air hoses"
    "Hose Rate Contract",
    "Civil Piping",             # "Pipe" SN → corpus: "civil and piping job"
    "DM Plant Piping",
    "Seamless Pipe",            # "Seamless Pipe" SN match
    "Pipe Fittings",            # Stage 2 corpus
    "Valve Servicing",          # "Valve" SN → corpus: "valve servicing sor"
    "Main Valve",
    "Gasket Rate Contract",     # "Gasket" SN → corpus: "rate contract for gaskets"
    "Metallic Gasket",
    "Oil Filter",               # "Filter" SN → corpus: "oil filter", "magnetic filter"
    "Filter Element",           # "Filter Element" / "Fee Filter" SN match
    "RO Filter Element",
    "MOV Spares",               # "Spares" SN → corpus: "mov spares", "du spares"
    "DU Spares",
    "OEM Spares",               # "Oem" SN → corpus: "oem spares", "oem visit"
    "Rotary Contract",          # "Rotary" SN → corpus: "rotary main contract"
    "Condition Monitoring Contract",
    "Scaffolding",              # "Scaffold" SN match
    "Crane Hire",               # "Hire Crane"/"Crane" SN → corpus: "hiring of crane"
    "Bearing Rate Contract",    # "Bearing" SN → corpus: "rate contract for bearings"
    "Roller Bearing",
    "Air End Assembly",         # "Assembly" SN → corpus: "air end assembly spares"
    "Fall Arrestor",            # "Fall Arrestor"/"Structural" SN match
    "Safety Harness",
    "Ball Valve",               # "Ball Valve" SN match
    "Strainer",                 # "Strainer" SN match
    "Tube Bundle",              # "Tube Bundle" SN match
    "Exchanger Tube",           # "Exchanger Tube" SN match
    "Stud Bolt",                # "Stud Bolt" SN match
    "Structural Items",         # "Structural"/"Structural Plate" SN match
    "Structural Plate",
    "Structural Steel",
    "Insulation Survey",        # "Insulation" SN → corpus: "insulation survey"
    "Thermal Survey",
    "Gear Box",                 # "Gear" SN → corpus: "gear box complete unit"
    "Blower Spares",            # "Blower" SN → corpus: "combustion air blower spares"
    "Rotor Balancing",          # "Rotor" SN → corpus: "dynamic balancing of rotors"
    "Coupling Spares",          # "Couple" SN → corpus: "erc coupling repair"
    "Forklift",                 # "Forklift" SN match
    "Telescopic Manlift",       # "Telescopic Conveyor" SN → corpus
    "Spring Hanger",            # "Spring" SN → corpus: "inspection of spring hangers"
    "Clamp",                    # "Clamp" SN → corpus: "online leak sealing clamps"
    "Online Leak Sealing",
    "Hydrotesting",             # "Hydrotesting" SN match
    "Pigging",                  # "Pig" SN → corpus: "pigging work lpg line"
    "Pneumatic Spares",         # "Pneumatic" SN → corpus: "pneumatic spares"
    "Cartridge Filter",         # "Cartridge" SN → corpus: "nas filtration cartridges"
    "NAS Filtration",
    "Steam Trap",               # "Steam Trap" SN match
    "Refractory",               # "Refractory" SN match
    "Ladder",                   # "Ladder" SN → corpus: "moveable ladder", "tower ladder"
    "Swing Ladder",             # "Swing Ladder" SN match
    "Trolley",                  # "Trolley" SN → corpus: "trolley ladder", "emergency trolley"
    "Manhole Cover",            # "Manhole Cover" SN match
    "Dome Cover",
    "Breakaway Coupling",       # "Breakaway Couple" SN match
    "Vapour Coupler",
    "Composite Works",          # "Composite" SN → corpus: "composite works"
    "Heater Service",           # "Heater" SN → corpus: "service contract for heaters"
    "Reactor Spares",           # "Reactor" SN → corpus: "spare reactor"
    "Tank Fabrication",         # "Tank Fabrication" SN match
    "Mobile Prover",            # "Mobile Prover" SN match
    "Grease",                   # "Grease" SN → corpus: "conkote grease"
    "Tyre Supply",              # "Tyre" SN → corpus: "tyres for refuellers"
    "Carousal Spares",          # "Carousal" SN match
    "Conveyor Spares",          # "Conveyor" (non-LPG) SN match
    ],
 
    "Civil & Works Contracts": [
        # Driveway — ~122 rows
        "Driveway Development",
        "RCC Driveway",
        "Driveway Job",
        # Road — ~138 rows
        "RCC Road",
        "Road Revamping",
        "Road Repair",
        "Bitumen Laying",
        "Road Surface",
        "Sealcoat",
        # Laying works — ~104 rows
        "RCC Laying",
        "Mastic Laying",
        "Bitumen Mastic",
        "Mainline Laying",
        # Solar — ~142 rows
        "Solar Panel",
        "Solar Installation",
        "Solar Street Light",
        "Solar Projects",
        "Solar PFR",
        "Solar Consultancy",
        # Building — ~62 rows
        "Building Maintenance",
        "Building Repair",
        "Building Occupancy",
        # Modernisation — ~113 rows
        "Modernization Job",
        "RO Modernization",
        # Site supervision — ~134 rows
        "Site Supervision",
        "Site Monitoring",
        "Site Survey",
        "Contour Survey",
        "Civil Works",          # "Civil" SN → corpus: "civil m&r", "civil jobs"
    "Civil Repair",
    "Civil Maintenance",    # "Civil Maint" SN match
    "Civil Standing Order", # "Civil Stand" SN match
    "Civil SOR",            # "Civil Sor"/"Civl Sor" SN match
    "Painting SOR",         # "Paint" SN → corpus: "painting sor"
    "Painting Works",
    "Painting Structure",   # "Paint Structure" SN match
    "RCC Road",             # "Road" SN → corpus: "rcc road", "road repair"
    "Road Repair",
    "Fit For Road",         # corpus match
    "Site Supervision",     # "Site" SN → corpus: "site supervision"
    "Site Survey",
    "RCC Laying",           # "Lay" SN → corpus: "rcc laying", "mastic laying"
    "Bitumen Mastic",
    "Mainline Laying",
    "Driveway Development", # "Driveway" SN match
    "RCC Driveway",
    "Boundary Wall",        # "Wall" SN → corpus: "boundary wall"
    "Civil Wall",
    "Building Repair",      # "Build" SN → corpus: "repair of civil building"
    "Civil Building",
    "Roof Sheet",           # "Roof" SN → corpus: "roof sheet construction"
    "Roof Repair",
    "Modernization Job",    # "Modernization"/"Modernisation" SN match
    "Drain Civil",          # "Drain" SN → corpus: "pre cast drain civil work"
    "Drain Modification",
    "Interior Works",       # "Interior" SN → corpus: "civil and interior renovation"
    "Guest House Renovation",
    "Mastic Flooring",      # "Mastic Floor" SN match
    "Floor Sweeping",
    "Waterproofing",        # "Proof" SN → corpus: "water proofing"
    "Water Proofing",
    "Paver Block",          # "Paver" SN → corpus: "paver block works"
    "Paver Jobs",
    "Tank Installation",    # "Tank Inst"/"Tank Instt"/"Tank Hills" SN match
    "Tank Installation Hills",
    "Foundation Works",     # "Foundation" SN → corpus: "pump foundation"
    "Pump Foundation",
    "ROU Civil",            # "Rou Civil" SN match
    "Civil Structure",      # "Civil Structure" SN match
    "Civil M&R",            # corpus generic match
    "Weigh Bridge Civil",   # "Weigh Bridge" SN → corpus: "civil job weigh bridge"
    "Retaining Wall",
    "Dismantle Civil",      # "Dismantle" SN → corpus: "dismantling of civil structure"
    "Refurbishment",        # "Refurbishment" SN match
    "Major Refurbishment",
    "Architectural Services",# "Architectural"/"Architectrual"/"Architechtural" SN
    "Architectural Design",
    "Drawing Retail",       # "Draw"/"Drawing" SN → corpus: "drawing of retail outlets"
    "NOC Drawing",          # "Noc Drawing" SN match
    "DFR Services",         # "Dfr" SN → corpus: "preparation of dfr"
    "PMC Services",         # "Pmc" SN match
    "Survey Services",      # "Survey Biomass"/"Detail Route" SN → corpus
    "Route Survey",
    "Topographical Survey",
    "Land Development",     # "Development" SN → corpus: "land development"
    "Driveway Civil",
    "Civil Reb",            # "Civil Reb" SN match
    "Major Civil",          # "Major Civil" SN match
    "Canal Civil",
    "Earthworks",           # "Earth" → corpus civil-side
    "Water Tank Civil",
    "Sewage Treatment",     # "Treatment" SN → corpus: "sewage treatment plant"
    "ETP Civil",
    "Soil Investigation",   # "Soil" SN → corpus: "soil survey"
    "Soil Survey",
    "Tank Civil",           # "Tank Jobs"/"Tank Inst" SN match
    "Tank Inspection",
    "Tank Repair",
    "Tank Pad",             # "Pad" SN → corpus: "tank pad sealing"
    "Tank Bottom",
    ],
 
    "Corporate Services": [
        # Car hire — ~195 rows
        "Car Hire",
        "Car Hiring Services",
        "Vehicle Hire",
        "Cab Hire",
        # Branding / flex — ~151 + 93 rows
        "Branding Work",
        "Flex Branding",
        "Flex Banner",
        "Banner Work",
        "Stickering",
        "SOP Boards",
        # Happy Shop — ~127 rows
        "Happy Shop",
        "Happy Shop Development",
        # PESO — ~164 rows (non-LPG)
        "PESO Work",
        "PESO Compliance",
        # Air quality / environment — ~16 rows
        "Air Quality Testing",
        "AAQMS",
        "Indoor Air Quality",
        # Cleaning / hygiene
        "Cleaning Chemical",
        "Drum Cleaning",
        "Hygiene Items",
        "Monsoon Protection",
        # Plastic / waste — ~126 rows
        "Plastic Waste Disposal",
        "Used Plastic Recycling",
        # Rate contracts / SOR (marketing services)
        "SOR Services",
        "ARC Services",
        "Rate Contract",
        "Car Hire",             # "Car Hire" SN match
    "Car Hiring",
    "Taxi Services",        # "Taxi"/"Taxi Hire" SN match
    "Taxi Hiring",
    "Cab Hire",             # "Cab" SN → corpus: "hiring of cabs"
    "Vehicle Hiring",       # "Hire Vehicle"/"Vehicle" SN match
    "Vehicle Hire",
    "Bus Hire",             # "Bus" SN → corpus: "hiring of tourist bus"
    "Bus Transport",
    "Car Driver",           # "Car Driver"/"Driver" SN match
    "Driver Services",
    "Company Car Driver",
    "Branding Work",        # "Brand" SN → corpus: "branding work", "flex branding"
    "Flex Branding",
    "Flex Banner",          # "Flex Banner"/"Flex" SN match
    "Flex Works",
    "Hoarding Works",       # "Hoarding"/"Hoard" SN match
    "Happy Shop",           # "Happy Shop" SN match
    "Happy Shop Development",
    "Apna Ghar",            # "Apna Ghar" SN match
    "Apna Ghar Signages",
    "SOP Boards",           # "Sop Board"/"Board" SN match
    "SOP Items",
    "ACP Board",
    "Signages",             # "Signages" SN match
    "Standee Banner",       # "Standee Brand" SN match
    "Grass Cutting",        # "Grass Cut" SN match
    "Grass Cutting Contract",
    "Garden Development",   # "Garden" SN → corpus: "garden development"
    "Landscaping",
    "Pest Control",         # "Pest" SN match
    "Janitorial Services",  # "Janitorial" SN match
    "Housekeeping Contract",# "House Keeping" SN match
    "Housekeeping Works",
    "Toilet Block",         # "Toilet" SN → corpus: "toilet block", "upkeep of toilets"
    "Caretaker Services",   # "Caretaker"/"Caretaking" SN match
    "Caretaker Cook",
    "Environmental Monitoring",# "Environment Monitor" SN match
    "Environmental Testing",
    "GreenCo Certification",# "Greenco"/"Greenco Implementation" SN match
    "GreenCo Rating",
    "ISO Certification",    # "Iso" SN → corpus: "iso certification"
    "IMS Certification",
    "Energy Audit",         # "Energy" SN → corpus: "energy audit"
    "Energy Management",
    "ERDMP Certification",  # "Erdmp" SN → corpus: "erdmp audit"
    "QRA HAZOP",            # "Qra Hazop"/"Qra Hira" SN match
    "HAZOP Study",
    "Event Management",     # "Event" SN → corpus: "event management"
    "HP Power Lab",
    "Food Supply",          # "Snack"/"Tea Coffee" SN → corpus
    "Tea Snacks",
    "Cafeteria Items",      # "Cafeteria" SN match
    "Diwali Sweets",        # "Diwali Sweet"/"Diwali" SN match
    "Diwali Hamper",
    "Dry Fruits",           # "Dry Fruit" SN match
    "Diwali Gifts",
    "Uniform Supply",       # "Uniform"/"Csa Uniform" SN match
    "CSA Uniform",
    "Printer Supply",       # "Printer" SN → corpus: "printers scanner"
    "Cartridge Supply",
    "Photocopier",          # "Photocopier" SN match
    "Printed Stationery",   # "Print"/"Stationery" SN → corpus
    "Stationary Items",     # "Stationary"/"Stationary Items" SN match
    "A4 Paper",             # "Size Paper" SN → corpus: "a4 size paper"
    "File Compactor",       # "File" SN → corpus: "file compactor storage"
    "Medical Checkup",      # "Camp"/"Medical"/"Health Checkup" SN match
    "Health Camp",
    "Health Checkup",
    "Occupational Health",
    "Colony Upkeep",        # "Colony" SN → corpus: "colony upkeep"
    "Housing Colony",
    "Drinking Water",       # "Drink" SN → corpus: "drinking water"
    "Water Tanker Supply",
    "Borewell Water",       # "Borewell" SN match
    "Potable Water",        # "Potable" SN match
    "Swachhta Items",       # "Swachhta Pakhwada" SN match
    "CSR Items",
    "Awards Application",   # "Award" SN → corpus: "golden peacock award"
    "Safety Award",
    "Office Assistance",    # "Office Assistance"/"Assistant" SN match
    "Office Assistant",
    "Data Entry",           # "Datum Entry" SN → corpus: "data entry operator"
    "Clerical Manpower",    # "Clerical" SN match
    "Manpower Nomination",  # "Nomination" SN → corpus: "manpower nomination"
    "Manpower Payment",     # "Payment" SN → corpus: "manpower payment"
    "Caretaking Services",
    "Furniture",            # "Chair" SN → corpus: "revolving and visitor chairs"
    "Chair Supply",
    "Custom Clearance",     # "Custom Clearance" SN match
    "Air Consolidation",
    "Bunker Audit",         # "Bunker" SN → corpus: "bunker supplier annual audit"
    "PNG Registration",     # "Png Registration" SN match
    "DMA Services",
    "CDCMS Support",        # "Cdcms" SN match
    "PSARA Services",       # "Psara" SN match
    "DGR Security",         # "Guard" SN → corpus: "dgr guards"
    "Security Guard",
    "Surveillance",         # "Surveillance" SN match
    "Integrated Facility",  # "Integrate" SN → corpus: "integrated facility management"
    "Facility Management",
    "Travel Expense",       # "Time" SN → corpus: "air time charges"
    "Air Time Charges",
    "Lease Line",           # "Lease" SN → corpus: "mpls leased line"
    "MPLS Link",
    "VSAT Bandwidth",       # "Vsat Bandwidth" SN match
    "Manpower Services",    # "Manpower Comco"/"Manpower" SN match
    "Manpower Supply",
    "Technical Manpower",   # "Technical"/"Operational"/"Provide" SN match
    "Operational Manpower",
    "Providing Manpower",
    "Contract Manpower",    # "Nomination"/"Payment" corpus
    "Caretaker Services",
    "Reporting",            # "Report" SN → corpus: "advisory report"
    "License Renewal",      # "License" SN → corpus: "sap dcs license", "autocad"
    "SAP License",
    "Software License",
    "Draftsman Services",   # "Draftsman" SN match
    "Drawing Preparation",  # "Draw" SN → corpus: "preparation of drawings"
    "T4S Audit",            # "Audit Locations"/"Audit Compliance" SN match
    "OISD Audit",
    "TPI Services",         # "Tpi" SN → corpus: "tpi service"
    "Third Party Inspection",
    "Standing Order",       # "Stock"/"Stand" SN → corpus: "standing order"
    "Standing PR",
    ],
 
    "Fire & Safety": [
        # Fire extinguisher — ~84 rows
        "Fire Extinguisher Servicing",
        "Fire Extinguisher Services",
        "Fire Extinguisher Items",
        "Extinguisher Maintenance",
        # Fire engines — ~11 rows
        "Fire Engine Repair",
        "Fire Engine Servicing",
        "Fire Engine B-Check",
        "Safety Manpower",      # "Safety" SN → corpus: "safety mech maintenance manpower"
    "Safety Maintenance",
    "Fire Fighting Equipment",  # "Fight" SN → corpus: "fire fighting equipments"
    "Fire Fighting Manpower",
    "Fire System Maintenance",
    "Fire Extinguisher",    # "Extinguisher" SN match
    "Extinguisher Servicing",
    "Fire Engine",          # "Engine" SN → corpus: "b-check for fire engines"
    "Fire Engine Repair",
    "Foam System",          # "Foam" SN → corpus: "ss foam system items"
    "Foam Generator",
    "HVLR Monitor",         # "Hvlr" SN match
    "Proximity Suit",       # "Proximity Suit" SN match
    "Clean Agent System",   # "Clean Agent" SN match
    "Fire Alarm",           # "Alarm" → corpus: "fire alarm system"
    "Rim Seal Fire",        # "Rim" SN → corpus: "rim seal fire protection"
    "FE Servicing",         # "Fe" SN → corpus: "servicing of fe and other items"
    "Fire Truck Repair",    # "Truck" SN → corpus: "repairing of fire truck"
    "Hydrant Civil",        # "Hydrant" → corpus: "civil work for hydrants"
    "Water Monitor",        # "Monitor" → corpus: "sply of water cum foam monitor"
    "ERDMP Audit",          # corpus safety match
    "SRV TRV Testing",      # "Srv" SN → corpus: "srv trv testing"
    "PSV Spares",           # "Psv"/"Tsv" SN → corpus: "psv spares"
    "TSV Testing",
    ],
 
    "Packaging & Additives": [
        # Cartons — ~130 rows
        "Cartons",
        "Carton Supply",
        "Bottle Checkweigher",
        "Ink For Carton",
        "Carton Printer Ink",
        # PPCP Pails — ~102 rows
        "PPCP Pail",
        "Round Pails",
        "5 Ltr Pail",
        # Containers — ~79 rows
        "Lube Containers",
        "R-PET Container",
        "Container Supply",
        "IBC Container",
        "Carton Supply",        # "Carton" SN → corpus: "supply of cartons"
    "Bottle Checkweigher",
    "Ink Carton",
    "PPCP Pail",            # "Ppcp Pail" SN match
    "Round Pails",
    "Container Supply",     # "Container" SN → corpus: "r-pet container"
    "Lube Container",
    "Rate Contract Hygiene",# "Rc" SN → corpus: "rc supply of hygiene items"
    "Hygiene Items",
    "Monsoon Protection",
    "FSI Items",
    "Label Supply",         # "Label" SN match
    "Wooden Pallet",        # "Wooden Pallet" SN match
    "Empty Drum",           # "Drum" SN → corpus: "supply of empty drums"
    "Drum Handling",
    "Bitumen Drum",
    "Plastic Seals",        # "Plastic" SN → corpus: "plastic seals supply"
    "Used Plastic Recycling",
    "HDPE Drum",            # "Blow Mould" SN → corpus: "hdpe blow moulded drum"
    "HDPE Barrel",          # "Hdpe Barrel" SN match
    "Barrel Filling",
    "Jumbo Bag",            # "Bag" SN → corpus: "empty jumbo bags"
    "Packing Material",
    ],
 
    "Materials & Warehouse": [
        # Drums — ~67 rows
        "Bitumen Drum",
        "Empty Drum",
        "Drum Handling",
        "Drum Supply",
        # Container seals
        "Seals Supply",
        "Plastic Seals",
        "Tamper Seals",
        "Chemical Rate Contract",  # "Chemical" SN → corpus: "chemicals for bioprocess"
    "SWRO Chemicals",          # corpus match
    "Boiler Chemicals",        # "Boiler" SN → corpus: "chemicals for boilers"
    "Boiler Spares",
    "Caustic Soda",            # "Caustic Soda" SN match
    "Caustic Potash",
    "Lithium Hydroxide",       # "Lithium Hydroxide" SN match
    "Hydrogen Peroxide",       # "Hydrogen Peroxide" SN match
    "Red Dye",                 # "Red Dye" SN match
    "Oil Dye",
    "Upkeeping Services",      # "Upkeeping"/"Upkeep" SN match
    "Plant Upkeep",
    "Waste Disposal",          # "Waste" SN → corpus: "waste disposal"
    "Sludge Disposal",         # "Disposal Sludge" SN match
    "Garbage Disposal",
    "Plastic Waste Disposal",
    "Non Stock Items",         # "Stock" SN → corpus: "non-stock items"
    "Standing Order Non Stock",
    "Scrap",                   # "Cut" SN → corpus: "cutting and shifting of scrap"
    "Scrap Disposal",
    ],
 
    "Mechanical - Refinery Contracts": [
        # TA / Turnaround — ~143 rows
        "TA Service Contract",
        "Pre-TA Services",
        "Turnaround Contract",
        # SOR / ARC — ~231 + 114 rows
        "SOR",
        "ARC",
        "Annual Rate Contract",
        "Rate Contract SOR",
        "Fabrication SOR",
        "Oil Bail Out",
        "SOR",                      # "Sor" SN match
    "SOR Works",
    "ARC",                      # "Arc" SN match
    "Annual Rate Contract",
    "TA Service",               # "Ta" SN → corpus: "ta service contract"
    "Turnaround Service",
    "Pre-TA Services",
    "TA Scaffolding",
    "Mechanical Assistance Contract",  # "Mech"/"Maint" SN → corpus
    "Mechanical SOR",
    "Plant Maintenance Contract",
    "SC",                       # "Sc" SN → corpus: "sc for water purifiers"
    "Service Contract",
    "WHA",                      # "Wha" SN → corpus: "wha 656", "wha items"
    "IBR Equipment",            # "Ibr" SN match
    "PDA TA",                   # "Pda Ta" SN match
    "LOUP TA",                  # "Loup Ta" SN match
    "OFCCU TA",                 # "Ta Ofccu" SN match
    "GFEC TA",                  # "Gfec Ta" SN match
    "NHGU TA",                  # "Ta Nhgu" SN match
    "DHDS Contract",            # "Dhds" SN → corpus: "dhds aru", "dhds sru"
    "Zonal Contract",
    "FRE TA",                   # "Fre" SN → corpus: "fre ta", "shutdown fre"
    "CDU TA",                   # "Cdu" SN → corpus: "cdu ii merox ta"
    "LOUP Contract",
    "TA LLI",                   # "Ta Lli" SN match
    "ISBL Contract",            # "Isbl Pipe" SN match
    "Tank Upkeeping",           # "Upkeeping" SN → corpus: "tank farm equipment upkeeping"
    "Upkeeping Services",
    ],
 
    "Laboratory": [
        # Testing — ~225 rows
        "Oil Filtration Testing",
        "Oil Testing",
        "Calibration Testing",
        "TR Oil Testing",
        # Calibration tanks / W&M — ~103 rows
        "Tank Calibration",
        "Gas Detector Calibration",
        "W&M Calibration",
        "Glassware Calibration",
    ],
 
    "Technical Process / CES / MES": [
        # Liaisoning (non-LPG) — engineering/statutory
        "Liaisoning Services",
        "Stamping Services",
        "CNG Liaisoning",
        "Peso Liaisoning",
    ]

})


SUBCAT_TO_CATEGORY_2 = {
    subcat: cat
    for cat, subcats in CATEGORY_MAP_2.items()
    for subcat in subcats
}
ALL_SUBCATS_2 = list(SUBCAT_TO_CATEGORY_2.keys())

_SUBCAT_TOKENS_2 = {sc: _tokenize_for_match(sc) for sc in ALL_SUBCATS_2}

def match_from_po_title_2(po_title: str):
    title_tokens = _tokenize_for_match(po_title)
    if not title_tokens:
        return 'Unmapped', 'Unmapped'

    best_subcat = None
    best_key    = None

    for rank, subcat in enumerate(ALL_SUBCATS_2):
        sc_tokens = _SUBCAT_TOKENS_2[subcat]
        if not sc_tokens:
            continue
        if not sc_tokens.issubset(title_tokens):
            continue
        key = (len(sc_tokens), -rank)
        if best_key is None or key > best_key:
            best_key    = key
            best_subcat = subcat

    if best_subcat is None:
        return 'Unmapped', 'Unmapped'
    return best_subcat, SUBCAT_TO_CATEGORY_2[best_subcat]


# ── Only run on rows that are still unmapped
unmapped_mask = df['Category'] == 'Unmapped'
print(f"Unmapped rows before second pass: {unmapped_mask.sum():,}")

results_2 = df.loc[unmapped_mask, TITLE_COL].apply(match_from_po_title_2)

df.loc[unmapped_mask, 'Sub Category'] = results_2.apply(lambda x: x[0])
df.loc[unmapped_mask, 'Category']     = results_2.apply(lambda x: x[1])

still_unmapped = (df['Category'] == 'Unmapped').sum()
newly_mapped   = unmapped_mask.sum() - still_unmapped
print(f"Newly mapped in second pass : {newly_mapped:,}")
print(f"Still unmapped              : {still_unmapped:,}")

Unmapped rows before second pass: 36,308
Newly mapped in second pass : 16,918
Still unmapped              : 19,390


In [24]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

df['_sort_group'] = (df['Category'] == 'Unmapped').astype(int)  # 0 = mapped, 1 = unmapped

df_final = df.sort_values(
    ['_sort_group', 'Category', 'Sub Category', 'Standard Name', 'Cluster ID'],
    ascending=[True, True, True, True, True]
).drop(columns=['_sort_group']).reset_index(drop=True)

df_mapped   = df_final[df_final['Category'] != 'Unmapped']
df_unmapped = df_final[df_final['Category'] == 'Unmapped']

# ── Colour palette: each category gets 2 alternating fills for its sub-categories
CATEGORY_COLOUR_PAIRS = [
    ('C6EFCE', 'A9D18E'),  # green tones
    ('BDD7EE', '9DC3E6'),  # blue tones

]
UNMAPPED_COLOUR = ('F2F2F2', 'D9D9D9')  # grey for unmapped

# Build category → colour pair index
unique_cats   = df_mapped['Category'].unique().tolist()
cat_colour_idx = {cat: i % len(CATEGORY_COLOUR_PAIRS) for i, cat in enumerate(unique_cats)}

# ── Style constants
HDR_FILL = PatternFill('solid', start_color='1F4E79')
HDR_FONT = Font(name='Arial', bold=True, color='FFFFFF', size=11)
DAT_FONT = Font(name='Arial', size=10)
CLR_FONT = Font(name='Arial', bold=True, size=10, color='1F4E79')
NM_FONT  = Font(name='Arial', italic=True, size=10, color='2E6DA4')
thin     = Side(style='thin', color='B0C4DE')
BORDER   = Border(left=thin, right=thin, top=thin, bottom=thin)
CENTER   = Alignment(horizontal='center', vertical='center')
WRAP     = Alignment(horizontal='left', vertical='center', wrap_text=True)
LEFT     = Alignment(horizontal='left', vertical='center')

output_cols = [TITLE_COL] + [c for c in EXTRA_COLS if c in df_final.columns]
HEADERS     = ['Cluster ID', 'Standard Name', 'Category', 'Sub Category'] + output_cols

wb = Workbook()
ws = wb.active
ws.title = "Clustered PO Titles"

# Header row
for c, h in enumerate(HEADERS, 1):
    cell = ws.cell(row=1, column=c, value=h)
    cell.font      = HDR_FONT
    cell.fill      = PatternFill('solid', start_color='1F4E79')
    cell.border    = BORDER
    cell.alignment = CENTER
ws.row_dimensions[1].height = 22

# ── Track sub-category alternation per category
prev_cat    = None
prev_subcat = None
subcat_toggle = 0          # flips 0/1 each time sub-category changes within a category

for r_idx, row in df_final.iterrows():
    excel_row = r_idx + 2
    cat    = row['Category']
    subcat = row['Sub Category']

    if cat == 'Unmapped':
        colour_pair = UNMAPPED_COLOUR
        # alternate grey shades by sub-category within unmapped
        if subcat != prev_subcat:
            subcat_toggle = 1 - subcat_toggle
            prev_subcat   = subcat
        fill_hex = colour_pair[subcat_toggle]

    else:
        pair_idx    = cat_colour_idx[cat]
        colour_pair = CATEGORY_COLOUR_PAIRS[pair_idx]

        if cat != prev_cat:
            # new category — reset toggle, treat first subcat as toggle=0
            subcat_toggle = 0
            prev_subcat   = subcat
            prev_cat      = cat
        elif subcat != prev_subcat:
            # same category, new sub-category — flip toggle
            subcat_toggle = 1 - subcat_toggle
            prev_subcat   = subcat

        fill_hex = colour_pair[subcat_toggle]

    row_fill = PatternFill('solid', start_color=fill_hex)

    for c, col in enumerate(HEADERS, 1):
        val  = row.get(col, '')
        cell = ws.cell(row=excel_row, column=c, value=val)
        cell.border = BORDER
        cell.fill   = row_fill
        if col == 'Cluster ID':
            cell.font = CLR_FONT; cell.alignment = CENTER
        elif col == 'Standard Name':
            cell.font = NM_FONT;  cell.alignment = WRAP
        elif col in ('Category', 'Sub Category'):
            cell.font = DAT_FONT; cell.alignment = WRAP
        elif col == TITLE_COL:
            cell.font = DAT_FONT; cell.alignment = WRAP
        else:
            cell.font = DAT_FONT; cell.alignment = LEFT
    ws.row_dimensions[excel_row].height = 18

# Column widths
col_widths = [12, 35, 25, 30, 55] + [22] * (len(output_cols) - 1)
for i, w in enumerate(col_widths, 1):
    ws.column_dimensions[get_column_letter(i)].width = w

ws.freeze_panes = 'A2'
ws.auto_filter.ref = f"A1:{get_column_letter(len(HEADERS))}1"

OUTPUT_FILE_V2 = 'Clustered_Output_final_v2.xlsx'
wb.save(OUTPUT_FILE_V2)

print(f"Saved -> {OUTPUT_FILE_V2}")
print(f"Mapped rows   : {len(df_mapped):,}")
print(f"Unmapped rows : {len(df_unmapped):,}")
print(f"Categories    : {df_mapped['Category'].nunique()}")

Saved -> Clustered_Output_final_v2.xlsx
Mapped rows   : 35,889
Unmapped rows : 19,390
Categories    : 26
